# Project Setup for Colab and Kaggle

This notebook was automatically bundled for cloud execution. Run the cell below to reconstruct the project structure and install dependencies.

In [ ]:
# =========================================================
# CLOUD ENVIRONMENT SETUP (AUTO-GENERATED)
# =========================================================
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if IN_COLAB or IN_KAGGLE:
    print("Running in Cloud Environment")
    
    # Write supporting files
    FILES = {
        'config.py': "from pathlib import Path\nfrom dataclasses import dataclass, field\nfrom typing import Optional, Any\n\n@dataclass\nclass FlowMatchingPredictorConfig:\n    hidden_size: int = 256\n    intermediate_size: int = 768\n    num_hidden_layers: int = 4\n    num_attention_heads: int = 8\n    hidden_act: str = 'silu'\n    rms_norm_eps: float = 1e-06\n    attention_bias: bool = True\n    attention_dropout: float = 0.1\n    mlp_bias: bool = True\n    track_dimensionality: int = 3\n    global_cond_dim: int = 512\n    head_dim: Optional[int] = None\n\n    def __post_init__(self) -> None:\n        if self.head_dim is None:\n            self.head_dim = self.hidden_size // self.num_attention_heads\n\n@dataclass\nclass Config:\n    device: Any = 'cuda'\n    seed: int = 42\n    dataset_path: Path = Path('./dataset/humanml3d-subset')\n    output_path: Path = Path('./output')\n    checkpoint_dir: Path = Path('./checkpoints')\n    checkpoint_interval: int = 50\n    motion_dim: int = 271\n    num_joints: int = 22\n    joint_dim: int = 3\n    max_motion_length: int = 200\n    fps: int = 20\n    feature_dims: tuple = (slice(0, 3), slice(3, 69), slice(69, 201), slice(201, 267), slice(267, 271))\n    encoder_motion_dim: int = 271\n    encoder_text_dim: int = 512\n    encoder_text_proj_dim: int = 128\n    encoder_hidden_dim: int = 512\n    encoder_per_joint_dim: int = 512\n    encoder_num_layers: int = 2\n    encoder_num_joints: int = 22\n    encoder_text_scale: float = 1.0\n    encoder_dropout: float = 0.1\n    predictor_config: FlowMatchingPredictorConfig = field(default_factory=lambda: FlowMatchingPredictorConfig(hidden_size=64, intermediate_size=4 * 64, num_hidden_layers=3, num_attention_heads=8, hidden_act='silu', rms_norm_eps=1e-06, attention_bias=True, attention_dropout=0.1, mlp_bias=True, track_dimensionality=3, head_dim=None))\n    batch_size: int = 192\n    learning_rate: float = 0.0001\n    num_epochs: int = 400\n    weight_decay: float = 1e-05\n    gradient_clip: float = 1.0\n    ema_decay: float = 0.999\n    cfg_dropout: float = 0.1\n    use_fk: bool = True\n    rollout_prob_start: float = 0.0\n    rollout_prob_end: float = 0.5\n    rollout_integration_steps: int = 5\n    use_consistency_loss: bool = True\n    consistency_loss_weight: float = 1.0\n    horizon: int = 20\n    curriculum: Optional[list[dict[str, int]]] = field(default_factory=lambda: [{'horizon': 5, 'epochs': 50}, {'horizon': 10, 'epochs': 100}, {'horizon': 20, 'epochs': 200}, {'horizon': 40, 'epochs': 400}])\n    num_workers: int = 4\n    pin_memory: bool = True\n    num_inference_steps: int = 20\n    guidance_scale: float = 1.0\n    val_interval: int = 5\n    val_batches: int = 20\n    val_use_ema: bool = True\n    save_best_val: bool = True\n    enable_profiling: bool = False\n    timing_log_interval: int = 100\n    unit_length = 5\n\n    def __post_init__(self):\n        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)\n        self.output_path.mkdir(parents=True, exist_ok=True)\n        self.dataset_path.mkdir(parents=True, exist_ok=True)\n\n    def get_predictor_feature_size(self) -> int:\n        return self.encoder_per_joint_dim\n\n    def to_dict(self) -> dict:\n        return {k: str(v) if isinstance(v, Path) else v for k, v in self.__dict__.items()}",
        'models.py': "import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom pathlib import Path\nfrom typing import Optional, List, Tuple, Union, cast\nfrom config import Config, FlowMatchingPredictorConfig\nfrom transformers.activations import ACT2FN\nfrom utils.motion_utils import get_fk_offsets, sequence_joints_to_features, generated_positions_to_271d, FeatureNormalizer\n\nclass KinematicChainEncoder(nn.Module):\n\n    def __init__(self, model_dim: int) -> None:\n        super().__init__()\n        joint_to_chain = [0] * 22\n        joint_to_depth = [0] * 22\n        for d, j in enumerate([0, 2, 5, 8, 11]):\n            joint_to_chain[j], joint_to_depth[j] = (0, d)\n        for d, j in enumerate([1, 4, 7, 10], 1):\n            joint_to_chain[j], joint_to_depth[j] = (1, d)\n        for d, j in enumerate([3, 6, 9, 12, 15], 1):\n            joint_to_chain[j], joint_to_depth[j] = (2, d)\n        for d, j in enumerate([14, 17, 19, 21], 4):\n            joint_to_chain[j], joint_to_depth[j] = (3, d)\n        for d, j in enumerate([13, 16, 18, 20], 4):\n            joint_to_chain[j], joint_to_depth[j] = (4, d)\n        self.register_buffer('joint_to_chain', torch.tensor(joint_to_chain))\n        self.register_buffer('joint_to_depth', torch.tensor(joint_to_depth))\n        self.joint_to_chain: torch.Tensor\n        self.joint_to_depth: torch.Tensor\n        self.chain_emb = nn.Embedding(5, model_dim // 2)\n        self.depth_emb = nn.Embedding(8, model_dim // 2)\n\n    def forward(self, joint_ids: torch.Tensor) -> torch.Tensor:\n        chains = self.joint_to_chain[joint_ids]\n        depths = self.joint_to_depth[joint_ids]\n        return torch.cat([self.chain_emb(chains), self.depth_emb(depths)], dim=-1)\n\nclass MotionHistoryEncoder(nn.Module):\n\n    def __init__(self, frame_feature_dim: int, text_embedding_dim: int, text_proj_dim: int, model_dim: int, per_joint_out_dim: int, num_layers: int=2, joint_count: int=22, text_scale: float=1.0, dropout: float=0.0, normalizer: Optional[FeatureNormalizer]=None) -> None:\n        super().__init__()\n        self.frame_feature_dim = frame_feature_dim\n        self.text_embedding_dim = text_embedding_dim\n        self.text_proj_dim = text_proj_dim\n        self.model_dim = model_dim\n        self.per_joint_out_dim = per_joint_out_dim\n        self.num_layers = num_layers\n        self.joint_count = joint_count\n        self.text_scale = text_scale\n        self.normalizer = normalizer\n        self.text_to_hidden = nn.Linear(text_embedding_dim, model_dim)\n        self.text_proj = nn.Linear(text_embedding_dim, text_proj_dim)\n        self.gru = nn.GRU(input_size=frame_feature_dim + text_proj_dim, hidden_size=model_dim, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)\n\n    def init_hidden(self, text_emb: torch.Tensor) -> torch.Tensor:\n        h0 = self.text_to_hidden(text_emb)\n        h0 = h0.unsqueeze(0).repeat(self.num_layers, 1, 1)\n        return h0\n\n    def _gru_block(self, motion_in: torch.Tensor, text_emb: torch.Tensor, h: Optional[torch.Tensor]) -> Tuple[torch.Tensor, torch.Tensor]:\n        B, T_step, _ = motion_in.shape\n        if h is None:\n            h = self.init_hidden(text_emb)\n        text_proj = self.text_proj(text_emb)\n        t_rep = self.text_scale * text_proj\n        t_rep = t_rep.unsqueeze(1).expand(B, T_step, -1)\n        gru_in = torch.cat([motion_in, t_rep], dim=-1)\n        h_seq, h_next = self.gru(gru_in, h)\n        h_t = h_seq[:, -1, :]\n        history_features = h_t.unsqueeze(1).expand(B, self.joint_count, self.model_dim)\n        return (history_features, h_next)\n\n    def forward(self, motion_seq: torch.Tensor, text_emb: torch.Tensor) -> torch.Tensor:\n        history_features, _ = self._gru_block(motion_seq, text_emb, h=None)\n        return history_features\n\n    def gru_step(self, x_t: torch.Tensor, text_emb: torch.Tensor, h: Optional[torch.Tensor], use_normalization: bool=False) -> Tuple[torch.Tensor, torch.Tensor]:\n        if use_normalization and self.normalizer is not None:\n            x_t = self.normalizer.normalize(x_t)\n        motion_in = x_t.unsqueeze(1)\n        history_features, h_next = self._gru_block(motion_in, text_emb, h)\n        return (history_features, h_next)\n\n    @property\n    def output_dim(self) -> int:\n        return self.per_joint_out_dim\n\nclass SinusoidalEmbedder(nn.Module):\n\n    def __init__(self, hidden_size: int, frequency_embedding_size: int=256):\n        super().__init__()\n        self.frequency_embedding_size = frequency_embedding_size\n        self.mlp: nn.Sequential = nn.Sequential(nn.Linear(frequency_embedding_size, hidden_size, bias=True), nn.SiLU(), nn.Linear(hidden_size, hidden_size, bias=True))\n\n    @staticmethod\n    def timestep_embedding(t: torch.Tensor, dim: int, max_period: int=10000) -> torch.Tensor:\n        half = dim // 2\n        if half == 0:\n            return torch.zeros((t.shape[0], dim), device=t.device, dtype=torch.float32)\n        max_period_tensor = torch.tensor(max_period, device=t.device, dtype=torch.float32)\n        freqs = torch.exp(-torch.log(max_period_tensor) * torch.arange(half, dtype=torch.float32, device=t.device) / half)\n        args = t[:, None].float() * freqs[None]\n        embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)\n        if dim % 2:\n            embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)\n        return embedding\n\n    def forward(self, t: torch.Tensor) -> torch.Tensor:\n        t_freq = self.timestep_embedding(t, self.frequency_embedding_size)\n        out = self.mlp(t_freq)\n        return out\n\nclass SpatialTrackMLP(nn.Module):\n\n    def __init__(self, config: FlowMatchingPredictorConfig):\n        super().__init__()\n        self.hidden_size = config.hidden_size\n        self.intermediate_size = config.intermediate_size\n        self.gate_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=config.mlp_bias)\n        self.up_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=config.mlp_bias)\n        self.down_proj = nn.Linear(self.intermediate_size, self.hidden_size, bias=config.mlp_bias)\n        self.act_fn = ACT2FN['silu']\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))\n\nclass SpatialTrackLayer(nn.Module):\n\n    def __init__(self, config: FlowMatchingPredictorConfig):\n        super().__init__()\n        self.hidden_size = config.hidden_size\n        self.num_heads = config.num_attention_heads\n        self.head_dim = getattr(config, 'head_dim', config.hidden_size // config.num_attention_heads)\n        self.self_attn = nn.MultiheadAttention(embed_dim=config.hidden_size, num_heads=config.num_attention_heads, dropout=config.attention_dropout, bias=config.attention_bias, batch_first=True)\n        self.mlp = SpatialTrackMLP(config)\n        self.input_layernorm = nn.LayerNorm(config.hidden_size, eps=config.rms_norm_eps)\n        self.post_attention_layernorm = nn.LayerNorm(config.hidden_size, eps=config.rms_norm_eps)\n        self.adaln_linear = nn.Linear(config.hidden_size, 6 * config.hidden_size, bias=True)\n        self.adaln_modulation = nn.Sequential(nn.SiLU(), self.adaln_linear)\n\n    def forward(self, hidden_states: torch.Tensor, adaln_conditioning: torch.Tensor, output_attentions: bool=False, **kwargs) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:\n        residual = hidden_states\n        adaln_out = self.adaln_modulation(adaln_conditioning)\n        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = adaln_out.chunk(6, dim=-1)\n        normed = self.input_layernorm(hidden_states)\n        normed = normed * (1 + scale_msa.unsqueeze(1)) + shift_msa.unsqueeze(1)\n        attn_output, attn_weights = self.self_attn(query=normed, key=normed, value=normed, need_weights=output_attentions)\n        attn_output = gate_msa.unsqueeze(1) * attn_output\n        hidden_states = residual + attn_output\n        residual = hidden_states\n        normed = self.post_attention_layernorm(hidden_states)\n        normed = normed * (1 + scale_mlp.unsqueeze(1)) + shift_mlp.unsqueeze(1)\n        mlp_output = self.mlp(normed)\n        mlp_output = gate_mlp.unsqueeze(1) * mlp_output\n        hidden_states = residual + mlp_output\n        if output_attentions:\n            return (hidden_states, attn_weights)\n        else:\n            return (hidden_states, None)\n\nclass FlowMatchingPredictor(nn.Module):\n\n    def __init__(self, feature_size: int, config: FlowMatchingPredictorConfig, out_channels: Optional[int]=None, use_relative_shift: bool=True, **kwargs):\n        super().__init__()\n        self.out_channels = out_channels or config.track_dimensionality\n        self.use_relative_shift = use_relative_shift\n        input_dim = config.track_dimensionality + feature_size + (config.track_dimensionality if use_relative_shift else 0)\n        self.input_projection = nn.Linear(input_dim, config.hidden_size)\n        self.global_cond_projection = nn.Linear(config.global_cond_dim, config.hidden_size)\n        self.time_embedder = SinusoidalEmbedder(config.hidden_size)\n        self.kinematic_encoder = KinematicChainEncoder(config.hidden_size)\n        self.register_buffer('joint_ids', torch.arange(22, dtype=torch.long))\n        self.kinematic_token_norm = nn.LayerNorm(config.hidden_size, eps=1e-06)\n        self.structural_layer_gates = nn.Parameter(torch.zeros(config.num_hidden_layers, dtype=torch.float32))\n        self.layers = nn.ModuleList([SpatialTrackLayer(config) for _ in range(config.num_hidden_layers)])\n        self.output_norm = nn.LayerNorm(config.hidden_size, eps=1e-06)\n        self.output_adaln_linear = nn.Linear(config.hidden_size, 2 * config.hidden_size, bias=True)\n        self.output_adaln = nn.Sequential(nn.SiLU(), self.output_adaln_linear)\n        self.output_projection = nn.Linear(config.hidden_size, self.out_channels)\n        self._initialize_weights()\n\n    def _initialize_weights(self):\n\n        def _basic_init(module):\n            if isinstance(module, nn.Linear):\n                torch.nn.init.xavier_uniform_(module.weight)\n                if module.bias is not None:\n                    nn.init.constant_(module.bias, 0)\n        self.apply(_basic_init)\n        nn.init.normal_(cast(nn.Linear, self.time_embedder.mlp[0]).weight, std=0.02)\n        nn.init.normal_(cast(nn.Linear, self.time_embedder.mlp[2]).weight, std=0.02)\n        for layer in self.layers:\n            if isinstance(layer, SpatialTrackLayer):\n                nn.init.constant_(layer.adaln_linear.weight, 0)\n                if layer.adaln_linear.bias is not None:\n                    nn.init.constant_(layer.adaln_linear.bias, 0)\n        nn.init.constant_(self.output_adaln_linear.weight, 0)\n        nn.init.constant_(self.output_adaln_linear.bias, 0)\n        nn.init.constant_(self.output_projection.weight, 0)\n        nn.init.constant_(self.output_projection.bias, 0)\n\n    def forward(self, noised_tracks: torch.Tensor, timesteps: torch.Tensor, text_embedding: torch.Tensor, track_features: torch.Tensor, prev_relative_shifts: Optional[torch.Tensor]=None, output_attentions: bool=False, output_hidden_states: bool=False, **kwargs) -> tuple[torch.Tensor, Optional[List[torch.Tensor]], Optional[List[torch.Tensor]]]:\n        features_to_concat = [noised_tracks, track_features]\n        if self.use_relative_shift:\n            if prev_relative_shifts is None:\n                prev_relative_shifts = torch.zeros_like(noised_tracks)\n            features_to_concat.append(prev_relative_shifts)\n        concatenated_features = torch.cat(features_to_concat, dim=-1)\n        hidden_states = self.input_projection(concatenated_features)\n        B, N, H = hidden_states.shape\n        kin_tokens = self.kinematic_encoder(self.joint_ids)\n        if kin_tokens.shape[0] != N:\n            raise ValueError(f'Joint count mismatch: predictor got N={N}, but kinematic table has {kin_tokens.shape[0]} joints.')\n        kin_tokens = self.kinematic_token_norm(kin_tokens)\n        kin_tokens = kin_tokens.to(device=hidden_states.device, dtype=hidden_states.dtype)\n        kin_tokens = kin_tokens.unsqueeze(0).expand(B, N, H)\n        time_cond = self.time_embedder(timesteps.squeeze(-1) if timesteps.dim() > 1 else timesteps)\n        global_cond_proj = self.global_cond_projection(text_embedding)\n        adaln_conditioning = time_cond + global_cond_proj\n        all_hidden_states: list[torch.Tensor] = []\n        all_self_attns: list[torch.Tensor] = []\n        for layer_idx, layer in enumerate(self.layers):\n            layer_gate = torch.tanh(self.structural_layer_gates[layer_idx]).to(dtype=hidden_states.dtype)\n            hidden_states = hidden_states + layer_gate * kin_tokens\n            layer_outputs = layer(hidden_states, adaln_conditioning=adaln_conditioning, output_attentions=output_attentions)\n            hidden_states = layer_outputs[0]\n            if output_hidden_states:\n                all_hidden_states.append(hidden_states)\n            if output_attentions:\n                all_self_attns.append(layer_outputs[1])\n        normed_states = self.output_norm(hidden_states)\n        shift, scale = self.output_adaln(adaln_conditioning).chunk(2, dim=-1)\n        normed_states = normed_states * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)\n        flow_prediction = self.output_projection(normed_states)\n        return (flow_prediction, all_hidden_states if output_hidden_states else None, all_self_attns if output_attentions else None)\n\nclass HumanMotionGenerator:\n\n    def __init__(self, encoder: MotionHistoryEncoder, predictor: FlowMatchingPredictor, config: Config) -> None:\n        super().__init__()\n        self.encoder = encoder\n        self.predictor = predictor\n        self.normalizer = encoder.normalizer\n        self.config = config\n\n    def eval(self) -> 'HumanMotionGenerator':\n        self.encoder.eval()\n        self.predictor.eval()\n        return self\n\n    def train(self, mode: bool=True) -> 'HumanMotionGenerator':\n        self.encoder.train(mode)\n        self.predictor.train(mode)\n        return self\n\n    def parameters(self):\n        for p in self.encoder.parameters():\n            yield p\n        for p in self.predictor.parameters():\n            yield p\n\n    def to(self, device):\n        self.encoder = self.encoder.to(device)\n        self.predictor = self.predictor.to(device)\n        return self\n\n    def generate_sequence(self, text: Union[str, List[str], torch.Tensor], num_frames: int=200, num_steps: int=10, horizon: int | None=None, input_positions: Optional[torch.Tensor]=None, total_duration: Optional[torch.Tensor]=None, guidance_scale: float=1.0, dataset_type: str='t2m', use_fk=True) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n        self.eval()\n        with torch.no_grad():\n            if isinstance(text, str):\n                from utils.text_encoder import CLIPEncoder\n                clip_encoder = CLIPEncoder()\n                text = clip_encoder(text)\n                B = 1\n            elif isinstance(text, list):\n                from utils.text_encoder import CLIPEncoder\n                clip_encoder = CLIPEncoder()\n                text: torch.Tensor = clip_encoder(text)\n                B = text.shape[0]\n            else:\n                if text.ndim != 3 or text.shape[1] != 1:\n                    raise ValueError(f'Pre-encoded text must have shape (B, 1, 512); got {tuple(text.shape)}')\n                B = text.shape[0]\n            device = next(self.parameters()).device\n            text = text.to(device=device)\n            if input_positions is None:\n                position_history = torch.zeros((B, 1, self.encoder.joint_count, self.predictor.out_channels), device=device)\n                feature_history = sequence_joints_to_features(position_history, dataset_type=dataset_type)\n            else:\n                input_positions = input_positions.to(device=device)\n                if input_positions.ndim == 3:\n                    seed_positions = input_positions.unsqueeze(1).clone()\n                elif input_positions.ndim == 4:\n                    seed_positions = input_positions.clone()\n                else:\n                    raise ValueError('input_positions must be shape (B, N, 3) or (B, T, N, 3)')\n                position_history = seed_positions\n                feature_history = sequence_joints_to_features(seed_positions, dataset_type=dataset_type)\n            fk_offsets = get_fk_offsets(position_history) if use_fk else None\n            feature_history = self.normalizer.normalize(feature_history) if self.normalizer is not None else feature_history\n            prev_relative_shifts = torch.zeros((B, 1, self.encoder.joint_count, self.predictor.out_channels), device=device)\n            if position_history.shape[1] > 1:\n                prev_relative_shifts = torch.cat([prev_relative_shifts, position_history[:, 1:] - position_history[:, :-1]], dim=1)\n            for frame_idx in range(num_frames):\n                current_positions = position_history[:, -1]\n                if horizon is not None:\n                    horizon_frames = min(horizon, feature_history.shape[1])\n                    encoder_input = feature_history[:, -horizon_frames:, :]\n                else:\n                    encoder_input = feature_history\n                text_emb = text[:, 0, :]\n                context_cond = self.encoder(encoder_input, text_emb)\n                x_t = torch.randn((B, self.encoder.joint_count, self.predictor.out_channels), device=device)\n                dt = 1.0 / num_steps\n                for step in range(num_steps):\n                    t = torch.full((B,), step * dt, device=device)\n                    relative_shifts = prev_relative_shifts[:, -1] if prev_relative_shifts.shape[1] > 0 else None\n                    flow_output = self.predictor.forward(track_features=context_cond, noised_tracks=x_t, timesteps=t, prev_relative_shifts=relative_shifts, text_embedding=text_emb, output_attentions=False, output_hidden_states=False)\n                    pred = flow_output[0]\n                    x_t = x_t + pred * dt\n                relative_shift = x_t\n                new_positions = current_positions + relative_shift\n                new_frame, _, fk_positions = generated_positions_to_271d(new_positions=new_positions, prev_positions=current_positions, dataset_type=dataset_type, normalizer=self.normalizer, fk_offsets=fk_offsets)\n                if fk_positions is not None:\n                    new_positions = fk_positions\n                    relative_shift = new_positions - current_positions\n                position_history = torch.cat([position_history, new_positions.unsqueeze(1)], dim=1)\n                feature_history = torch.cat([feature_history, new_frame.unsqueeze(1)], dim=1)\n                prev_relative_shifts = torch.cat([prev_relative_shifts, relative_shift.unsqueeze(1)], dim=1)\n                if (frame_idx + 1) % 50 == 0:\n                    print(f'Generated {frame_idx + 1}/{num_frames} frames')\n            return (position_history, feature_history, prev_relative_shifts)\n\n    @classmethod\n    def load_from_checkpoint(cls, checkpoint_path: Union[str, Path], config: Config, device: str='cpu', normalizer: Optional[FeatureNormalizer]=None) -> 'HumanMotionGenerator':\n        print(f'Loading checkpoint from {checkpoint_path}...')\n        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)\n        if 'config' in checkpoint:\n            config: Config = checkpoint['config']\n        encoder = MotionHistoryEncoder(frame_feature_dim=config.encoder_motion_dim, text_embedding_dim=config.encoder_text_dim, text_proj_dim=config.encoder_text_proj_dim, model_dim=config.encoder_hidden_dim, per_joint_out_dim=config.encoder_per_joint_dim, num_layers=config.encoder_num_layers, joint_count=config.encoder_num_joints, text_scale=config.encoder_text_scale, dropout=config.encoder_dropout, normalizer=normalizer).to(device)\n        predictor_config = config.predictor_config\n        feature_size = config.get_predictor_feature_size()\n        predictor = FlowMatchingPredictor(feature_size=feature_size, config=predictor_config, out_channels=None, use_relative_shift=True, normalizer=normalizer).to(device)\n        if 'encoder_ema' in checkpoint and 'predictor_ema' in checkpoint:\n            print('Loading EMA weights for generation...')\n            encoder.load_state_dict(checkpoint['encoder_ema'])\n            predictor.load_state_dict(checkpoint['predictor_ema'])\n        else:\n            print('Loading standard weights (EMA not found)...')\n            encoder.load_state_dict(checkpoint['encoder'])\n            predictor.load_state_dict(checkpoint['predictor'])\n        encoder.to(device)\n        predictor.to(device)\n        encoder.eval()\n        predictor.eval()\n        return cls(encoder, predictor, config)",
        'requirements.txt': '# Core ML dependencies\ntorch\ntorchvision\nnumpy\nscipy\ntransformers\n\n# Data processing\npandas\n\n# Visualization\nmatplotlib\nplotly\nimageio[ffmpeg]\nseaborn\n\n# Utilities\ntqdm\ngdown\n\n# Text Encoding\nftfy\nregex\n',
        'utils/__init__.py': '# Utils module for motion generation project\n',
        'utils/dataset.py': 'import torch\nimport numpy as np\nfrom os.path import join as pjoin\nimport random\nfrom tqdm import tqdm\nfrom torch.utils.data import Dataset, DataLoader\nfrom pathlib import Path\nfrom typing import List, Dict, Any, Optional, Tuple\nfrom config import Config\nfrom utils.motion_utils import FeatureNormalizer\n\nclass Text2MotionDataset(Dataset):\n\n    def __init__(self, config: Config, mean: np.ndarray, std: np.ndarray, split: str=\'train\'):\n        self.config = config\n        self.max_length = 20\n        self.pointer = 0\n        self.max_motion_length = config.max_motion_length\n        self.current_horizon = config.max_motion_length\n        min_motion_len = 40\n        motion_dir = config.dataset_path / \'new_joint_vecs\'\n        joints_dir = config.dataset_path / \'new_joints\'\n        text_dir = config.dataset_path / \'texts\'\n        split_file = config.dataset_path / f\'{split}.txt\'\n        data_dict = {}\n        id_list = []\n        with open(str(split_file), \'r\', encoding=\'utf-8\') as f:\n            for line in f.readlines():\n                id_list.append(line.strip())\n        new_name_list = []\n        length_list = []\n        for name in tqdm(id_list):\n            try:\n                motion = np.load(pjoin(str(motion_dir), name + \'.npy\'))\n                joints = np.load(pjoin(str(joints_dir), name + \'.npy\'))\n                if len(motion) < min_motion_len or len(motion) >= 200:\n                    continue\n                text_data = []\n                flag = False\n                with open(pjoin(str(text_dir), name + \'.txt\'), \'r\', encoding=\'utf-8\') as f:\n                    for line in f.readlines():\n                        text_dict: Dict[str, Optional[Any]] = {}\n                        line_split = line.strip().split(\'#\')\n                        caption = line_split[0]\n                        tokens = line_split[1].split(\' \')\n                        f_tag = float(line_split[2])\n                        to_tag = float(line_split[3])\n                        f_tag = 0.0 if np.isnan(f_tag) else f_tag\n                        to_tag = 0.0 if np.isnan(to_tag) else to_tag\n                        text_dict[\'caption\'] = caption\n                        text_dict[\'tokens\'] = tokens\n                        if f_tag == 0.0 and to_tag == 0.0:\n                            flag = True\n                            text_data.append(text_dict)\n                        else:\n                            try:\n                                n_motion = motion[int(f_tag * 20):int(to_tag * 20)]\n                                if len(n_motion) < min_motion_len or len(n_motion) >= 200:\n                                    continue\n                                new_name = random.choice(\'ABCDEFGHIJKLMNOPQRSTUVW\') + \'_\' + name\n                                while new_name in data_dict:\n                                    new_name = random.choice(\'ABCDEFGHIJKLMNOPQRSTUVW\') + \'_\' + name\n                                n_joints = joints[int(f_tag * 20):int(to_tag * 20)]\n                                data_dict[new_name] = {\'motion\': n_motion, \'joints\': n_joints, \'length\': len(n_motion), \'text\': [text_dict]}\n                                new_name_list.append(new_name)\n                                length_list.append(len(n_motion))\n                            except:\n                                print(line_split)\n                                print(line_split[2], line_split[3], f_tag, to_tag, name)\n                if flag:\n                    data_dict[name] = {\'motion\': motion, \'joints\': joints, \'length\': len(motion), \'text\': text_data}\n                    new_name_list.append(name)\n                    length_list.append(len(motion))\n            except Exception as e:\n                pass\n        name_length_pairs = list(zip(new_name_list, length_list))\n        name_length_pairs.sort(key=lambda x: x[1])\n        self.name_list = [pair[0] for pair in name_length_pairs]\n        self.length_arr = np.array([pair[1] for pair in name_length_pairs])\n        self.data_dict = data_dict\n        self.mean = torch.from_numpy(mean).float()\n        self.std = torch.from_numpy(std).float()\n        self.text_cache_path = config.dataset_path / \'text_embeddings_cache.pt\'\n        self.text_cache: Dict[str, torch.Tensor] = {}\n        if self.text_cache_path.exists():\n            print(f\'Loading text embedding cache from {self.text_cache_path}...\')\n            self.text_cache = torch.load(self.text_cache_path, weights_only=False)\n        all_captions = set()\n        for key, data in self.data_dict.items():\n            for text_item in data[\'text\']:\n                all_captions.add(text_item[\'caption\'])\n        missing_captions = [cap for cap in all_captions if cap not in self.text_cache]\n        if missing_captions:\n            print(f\'Computed {len(self.text_cache)}/{len(all_captions)} embeddings. Computing {len(missing_captions)} missing...\')\n            from utils.text_encoder import CLIPEncoder\n            clip_encoder = CLIPEncoder(model_name=\'openai/clip-vit-base-patch32\')\n            clip_encoder.to(config.device)\n            batch_size = 32\n            for i in tqdm(range(0, len(missing_captions), batch_size), desc=\'Encoding Texts\'):\n                batch_caps = missing_captions[i:i + batch_size]\n                with torch.no_grad():\n                    embeddings = clip_encoder(batch_caps).cpu()\n                for cap, emb in zip(batch_caps, embeddings):\n                    self.text_cache[cap] = emb\n            print(f\'Saving updated cache to {self.text_cache_path}...\')\n            torch.save(self.text_cache, self.text_cache_path)\n            del clip_encoder\n            torch.cuda.empty_cache()\n        else:\n            print(\'All text embeddings are cached.\')\n\n    def get_normalizer(self) -> FeatureNormalizer:\n        return FeatureNormalizer(mean=self.mean.clone(), std=self.std.clone())\n\n    def __len__(self):\n        return len(self.data_dict) - self.pointer\n    \'\\n    FINAL CORRECT __getitem__ implementation\\n    This is the ONLY version that works - replace everything else\\n    \'\n\n    def __getitem__(self, item):\n        idx = self.pointer + item\n        data = self.data_dict[self.name_list[idx]]\n        motion = data[\'motion\']\n        joints = data[\'joints\']\n        original_length = data[\'length\']\n        text_list = data[\'text\']\n        text_data = random.choice(text_list)\n        caption = text_data[\'caption\']\n        motion = torch.from_numpy(motion.copy()).float()\n        joints = torch.from_numpy(joints.copy()).float()\n        target_len = self.current_horizon\n        current_len = original_length\n        if current_len < target_len:\n            pad_size = target_len - current_len\n            motion = torch.cat([motion, torch.zeros(pad_size, motion.shape[1], dtype=motion.dtype, device=motion.device)], dim=0)\n            joints = torch.cat([joints, torch.zeros(pad_size, joints.shape[1], joints.shape[2], dtype=joints.dtype, device=joints.device)], dim=0)\n        elif current_len > target_len:\n            start_idx = random.randint(0, current_len - self.current_horizon)\n            motion = motion[start_idx:start_idx + self.current_horizon]\n            joints = joints[start_idx:start_idx + self.current_horizon]\n        assert motion.shape[0] == target_len, f\'Motion shape[0]={motion.shape[0]}, expected {target_len}\'\n        assert motion.shape[1] == 271, f\'Motion shape[1]={motion.shape[1]}, expected 271\'\n        assert joints.shape[0] == target_len, f\'Joints shape[0]={joints.shape[0]}, expected {target_len}\'\n        text_embedding = self.text_cache[caption]\n        if isinstance(text_embedding, np.ndarray):\n            text_embedding = torch.from_numpy(text_embedding).float()\n        else:\n            text_embedding = text_embedding.float()\n        if text_embedding.ndim == 1:\n            if text_embedding.shape[0] != CLIP_EMBED_DIM:\n                raise ValueError(f"Invalid 1D text embedding shape {tuple(text_embedding.shape)} for caption \'{caption}\'. Expected ({CLIP_EMBED_DIM},).")\n            text_embedding = text_embedding.unsqueeze(0)\n        elif text_embedding.ndim == 2:\n            if text_embedding.shape == (1, CLIP_EMBED_DIM):\n                pass\n            elif text_embedding.shape == (CLIP_MAX_SEQ_LEN, CLIP_EMBED_DIM):\n                raise ValueError(\'Detected legacy CLIP sequence embedding shape (77, 512) in text cache. Regenerate text_embeddings_cache.pt using pooled CLIP outputs (1, 512).\')\n            else:\n                raise ValueError(f"Invalid 2D text embedding shape {tuple(text_embedding.shape)} for caption \'{caption}\'. Expected (1, {CLIP_EMBED_DIM}).")\n        else:\n            raise ValueError(f"Invalid text embedding rank {text_embedding.ndim} for caption \'{caption}\'. Expected rank 2 with shape (1, 512).")\n        return (caption, motion, joints, original_length, text_embedding)\n\n    def reset_min_len(self, length: int | None=None):\n        if length is None:\n            self.pointer = 0\n            return\n        assert length <= self.max_motion_length\n        self.pointer = np.searchsorted(self.length_arr, length)\n        print(\'Pointer Pointing at %d\' % self.pointer)\n\n    def set_horizon(self, horizon: int | None=None):\n        if horizon is None:\n            self.current_horizon = self.max_motion_length\n            self.reset_min_len()\n            return\n        assert 1 <= horizon <= self.max_motion_length\n        self.current_horizon = horizon\n        self.reset_min_len(horizon)\nCLIP_MAX_SEQ_LEN = 77\nCLIP_EMBED_DIM = 512\n\ndef text2motion_collate_fn(batch: List[Tuple[str, torch.Tensor, torch.Tensor, int, torch.Tensor]]) -> Dict[str, Any]:\n    captions = [b[0] for b in batch]\n    motions_list = [b[1] for b in batch]\n    joints_list = [b[2] for b in batch]\n    lengths = [b[3] for b in batch]\n    text_embs_list = [b[4] for b in batch]\n\n    def to_tensor(x):\n        if isinstance(x, np.ndarray):\n            return torch.from_numpy(x).float()\n        elif isinstance(x, torch.Tensor):\n            return x.float()\n        else:\n            return torch.tensor(x).float()\n    motions_list = [to_tensor(x) for x in motions_list]\n    joints_list = [to_tensor(x) for x in joints_list]\n    text_embs_list = [to_tensor(x) for x in text_embs_list]\n    for idx, text_emb in enumerate(text_embs_list):\n        if text_emb.ndim != 2 or text_emb.shape[0] != 1 or text_emb.shape[1] != CLIP_EMBED_DIM:\n            raise ValueError(f\'Invalid text embedding at batch index {idx}: shape {tuple(text_emb.shape)}. Expected (1, {CLIP_EMBED_DIM}).\')\n    motion_batch = torch.stack(motions_list, dim=0)\n    joints_batch = torch.stack(joints_list, dim=0)\n    length_batch = torch.tensor(lengths, dtype=torch.long)\n    text_emb_batch = torch.stack(text_embs_list, dim=0)\n    return {\'captions\': captions, \'motion\': motion_batch, \'joints\': joints_batch, \'lengths\': length_batch, \'text_clip\': text_emb_batch}\n\ndef create_dataloader(config: Config, split: str=\'train\', shuffle: bool=True) -> Tuple[DataLoader, FeatureNormalizer]:\n    mean_path = config.dataset_path / \'Mean.npy\'\n    std_path = config.dataset_path / \'Std.npy\'\n    if not mean_path.exists() or not std_path.exists():\n        raise FileNotFoundError(f\'Mean.npy and/or Std.npy not found in {config.dataset_path}. Please ensure Mean.npy and Std.npy exist in the dataset directory.\')\n    mean = np.load(mean_path)\n    std = np.load(std_path)\n    dataset_obj = Text2MotionDataset(config, mean, std, split)\n    normalizer = FeatureNormalizer(mean=torch.from_numpy(mean).float(), std=torch.from_numpy(std).float())\n    dataloader = DataLoader(dataset_obj, batch_size=config.batch_size, shuffle=shuffle, num_workers=config.num_workers, pin_memory=config.pin_memory, collate_fn=text2motion_collate_fn)\n    return (dataloader, normalizer)\n\ndef load_sample(dataset_path: Path, file_id: str) -> Dict[str, Optional[Any]]:\n    features_path = dataset_path / \'new_joint_vecs\' / f\'{file_id}.npy\'\n    joints_path = dataset_path / \'new_joints\' / f\'{file_id}.npy\'\n    text_path = dataset_path / \'texts\' / f\'{file_id}.txt\'\n    data: Dict[str, Optional[Any]] = {\'file_id\': file_id}\n    if features_path.exists():\n        data[\'features\'] = np.load(features_path)\n    else:\n        print(f\'Warning: Features not found for {file_id}\')\n        data[\'features\'] = None\n    if joints_path.exists():\n        data[\'joints\'] = np.load(joints_path)\n    else:\n        print(f\'Warning: Joints not found for {file_id}\')\n        data[\'joints\'] = None\n    if text_path.exists():\n        with open(text_path, \'r\') as f:\n            descriptions = [line.strip().split(\'#\')[0] for line in f.readlines()]\n            data[\'text\'] = descriptions[0] if descriptions else \'\'\n    else:\n        data[\'text\'] = \'\'\n    return data',
        'utils/motion_utils.py': "import torch\nimport numpy as np\nfrom typing import List, Tuple, Dict, Any, Optional\nfrom utils.quaternion import qrot, qinv, qmul, quaternion_to_cont6d, cont6d_to_matrix, cont6d_to_quaternion\nT2M_RAW_OFFSETS = torch.tensor([[0, 0, 0], [1, 0, 0], [-1, 0, 0], [0, 1, 0], [0, -1, 0], [0, -1, 0], [0, 1, 0], [0, -1, 0], [0, -1, 0], [0, 1, 0], [0, 0, 1], [0, 0, 1], [0, 1, 0], [1, 0, 0], [-1, 0, 0], [0, 0, 1], [0, -1, 0], [0, -1, 0], [0, -1, 0], [0, -1, 0], [0, -1, 0], [0, -1, 0]], dtype=torch.float32)\nT2M_KINEMATIC_CHAIN = [[0, 2, 5, 8, 11], [0, 1, 4, 7, 10], [0, 3, 6, 9, 12, 15], [9, 14, 17, 19, 21], [9, 13, 16, 18, 20]]\n__PARENT_INDICES, __CHILD_INDICES = zip(*[(parent, child) for chain in T2M_KINEMATIC_CHAIN for parent, child in zip(chain[:-1], chain[1:])])\n_PARENT_INDICES = torch.tensor(__PARENT_INDICES, dtype=torch.long)\n_CHILD_INDICES = torch.tensor(__CHILD_INDICES, dtype=torch.long)\nDATASET_CONFIGS = {'t2m': {'name': 'HumanML3D', 'num_joints': 22, 'feature_dim': 271, 'raw_offsets': T2M_RAW_OFFSETS, 'kinematic_chain': T2M_KINEMATIC_CHAIN, 'face_joint_indx': [2, 1, 17, 16], 'fid_r': [8, 11], 'fid_l': [7, 10]}}\n\ndef get_dataset_config(dataset_type: str='t2m') -> Dict[str, Any]:\n    if dataset_type not in DATASET_CONFIGS:\n        raise ValueError(f'Unknown dataset_type: {dataset_type}. Available: {list(DATASET_CONFIGS.keys())}')\n    return DATASET_CONFIGS[dataset_type]\nFEATURE_SLICES = {'root_features': slice(0, 3), 'ric_positions': slice(3, 69), 'rotations_6d': slice(69, 201), 'local_velocities': slice(201, 267), 'foot_contacts': slice(267, 271)}\n\ndef get_fk_offsets(positions: torch.Tensor) -> torch.Tensor:\n    B, _, J, _ = positions.shape\n    parent_idx = _PARENT_INDICES.to(positions.device)\n    child_idx = _CHILD_INDICES.to(positions.device)\n    edge_lengths = torch.norm(positions[..., child_idx, :] - positions[..., parent_idx, :], dim=-1).mean(dim=1)\n    mean_lengths = torch.zeros(B, J, dtype=positions.dtype, device=positions.device)\n    mean_lengths[:, child_idx] = edge_lengths\n    unit_offsets = T2M_RAW_OFFSETS.to(positions.device)\n    return (unit_offsets * mean_lengths.unsqueeze(-1)).detach()\n\ndef _normalize_vector(v: torch.Tensor, eps: float=1e-10) -> torch.Tensor:\n    inv_norm = torch.rsqrt((v * v).sum(dim=-1, keepdim=True).clamp(min=eps))\n    return v * inv_norm\n\ndef _compute_ik(positions: torch.Tensor, raw_offsets: torch.Tensor, kinematic_chain: List[List[int]], face_joint_indx: List[int]) -> torch.Tensor:\n    batch_shape = positions.shape[:-2]\n    device = positions.device\n    dtype = positions.dtype\n    positions_flat = positions.reshape(-1, 22, 3)\n    B = positions_flat.shape[0]\n    l_hip, r_hip, sdr_r, sdr_l = face_joint_indx\n    across1 = positions_flat[:, r_hip] - positions_flat[:, l_hip]\n    across2 = positions_flat[:, sdr_r] - positions_flat[:, sdr_l]\n    across = across1 + across2\n    across = _normalize_vector(across)\n    forward = positions_flat.new_zeros(B, 3)\n    forward[:, 0] = across[:, 2]\n    forward[:, 2] = -across[:, 0]\n    forward = _normalize_vector(forward)\n    target = positions_flat.new_zeros(B, 3)\n    target[:, 2] = 1.0\n    root_quat = _qbetween(forward, target, assume_v0_normalized=True)\n    quaternions = torch.zeros(B, 22, 4, device=device, dtype=dtype)\n    quaternions[:, 0] = root_quat\n    offsets = raw_offsets.unsqueeze(0).expand(B, -1, -1)\n    offsets_norm = _normalize_vector(offsets)\n    qinv_sign = root_quat.new_tensor([1.0, -1.0, -1.0, -1.0]).view(1, 4)\n    for chain in kinematic_chain:\n        R = root_quat\n        for i in range(len(chain) - 1):\n            parent_idx = chain[i]\n            child_idx = chain[i + 1]\n            u = offsets_norm[:, child_idx]\n            v = positions_flat[:, child_idx] - positions_flat[:, parent_idx]\n            v = _normalize_vector(v)\n            rot_u_v = _qbetween(u, v, assume_v0_normalized=True, assume_v1_normalized=True)\n            R_loc = qmul(R * qinv_sign, rot_u_v)\n            quaternions[:, child_idx] = R_loc\n            R = qmul(R, R_loc)\n    return quaternions.reshape(batch_shape + (22, 4))\n\ndef _qbetween(v0: torch.Tensor, v1: torch.Tensor, assume_v0_normalized: bool=False, assume_v1_normalized: bool=False) -> torch.Tensor:\n    if not assume_v0_normalized:\n        v0 = _normalize_vector(v0)\n    if not assume_v1_normalized:\n        v1 = _normalize_vector(v1)\n    dot = (v0 * v1).sum(dim=-1, keepdim=True)\n    cross = torch.cross(v0, v1, dim=-1)\n    w = 1.0 + dot\n    q = torch.cat([w, cross], dim=-1)\n    q = _normalize_vector(q)\n    return q\n\ndef _forward_kinematics(rotations_6d: torch.Tensor, root_pos: torch.Tensor, offsets: torch.Tensor, kinematic_chain: List[List[int]]) -> torch.Tensor:\n    batch_shape = rotations_6d.shape[:-2]\n    device = rotations_6d.device\n    dtype = rotations_6d.dtype\n    rotations_flat = rotations_6d.reshape(-1, 22, 6)\n    root_pos_flat = root_pos.reshape(-1, 3)\n    B = rotations_flat.shape[0]\n    positions = torch.zeros(B, 22, 3, device=device, dtype=dtype)\n    positions[:, 0] = root_pos_flat\n    if offsets.ndim == 2:\n        offsets_expanded = offsets.unsqueeze(0).expand(B, -1, -1)\n    elif offsets.ndim == 3:\n        offsets_expanded = offsets\n    else:\n        raise ValueError(f'Offsets must have shape (22, 3) or (B, 22, 3), got {offsets.shape}')\n    rot_matrices = cont6d_to_matrix(rotations_flat)\n    for chain in kinematic_chain:\n        matR = rot_matrices[:, 0]\n        for i in range(1, len(chain)):\n            child_idx = chain[i]\n            parent_idx = chain[i - 1]\n            child_rot = rot_matrices[:, child_idx]\n            matR = torch.bmm(matR, child_rot)\n            offset_vec = offsets_expanded[:, child_idx].unsqueeze(-1)\n            positions[:, child_idx] = torch.bmm(matR, offset_vec).squeeze(-1) + positions[:, parent_idx]\n    return positions.reshape(batch_shape + (22, 3))\n\ndef subset_271d_to_72d(x: torch.Tensor) -> torch.Tensor:\n    slices = [slice(0, 1), slice(1, 3), slice(69, 75), slice(6, 69)]\n    x_72d_list = []\n    for sl in slices:\n        x_72d_list.append(x[..., sl])\n    x_72d = torch.cat(x_72d_list, dim=-1)\n    return x_72d\n\ndef _subset_unused(x: torch.Tensor) -> torch.Tensor:\n    root_height = x[..., 0:1]\n    root_vel = x[..., 1:3]\n    root_rot6d = x[..., 69:75]\n    prev_root = torch.cat([root_height, root_vel, root_rot6d], dim=-1)\n    joint_ric = x[..., 6:69]\n    joint_rot6d = x[..., 75:201]\n    joint_vel = x[..., 204:267]\n    prev_joints = torch.cat([joint_ric, joint_rot6d, joint_vel], dim=-1)\n    return torch.cat([prev_root, prev_joints], dim=-1)\n\nclass FeatureNormalizer:\n\n    def __init__(self, mean=torch.zeros(271), std=torch.ones(271)):\n        self.mean = mean\n        self.std = std\n\n    @classmethod\n    def load_from_files(cls, mean_path, std_path, device=torch.device('cpu')):\n        mean_np = np.load(mean_path)\n        std_np = np.load(std_path)\n        mean = torch.from_numpy(mean_np).float().to(device)\n        std = torch.from_numpy(std_np).float().to(device)\n        return cls(mean, std)\n\n    def normalize(self, features: torch.Tensor) -> torch.Tensor:\n        if features.shape[-1] != 271:\n            raise ValueError(f'Expected features to have shape (..., 271), got {features.shape}')\n        if features.device != self.mean.device:\n            self.mean = self.mean.to(features.device)\n            self.std = self.std.to(features.device)\n        return (features - self.mean) / self.std\n\n    def denormalize(self, features: torch.Tensor) -> torch.Tensor:\n        if features.shape[-1] != 271:\n            raise ValueError(f'Expected features to have shape (..., 271), got {features.shape}')\n        if features.device != self.mean.device:\n            self.mean = self.mean.to(features.device)\n            self.std = self.std.to(features.device)\n        return features * self.std + self.mean\n\n    def denormalize_flow_output(self, flow_output: torch.Tensor) -> torch.Tensor:\n        if flow_output.shape[-1] != 72:\n            raise ValueError(f'Expected flow_output to have shape (..., 72), got {flow_output.shape}')\n        if flow_output.device != self.mean.device:\n            self.mean = self.mean.to(flow_output.device)\n            self.std = self.std.to(flow_output.device)\n        mean_72d = subset_271d_to_72d(self.mean)\n        std_72d = subset_271d_to_72d(self.std)\n        return flow_output * std_72d + mean_72d\n\n    def normalize_flow_output(self, flow_output: torch.Tensor) -> torch.Tensor:\n        if flow_output.shape[-1] != 72:\n            raise ValueError(f'Expected flow_output to have shape (..., 72), got {flow_output.shape}')\n        if flow_output.device != self.mean.device:\n            self.mean = self.mean.to(flow_output.device)\n            self.std = self.std.to(flow_output.device)\n        mean_72d = subset_271d_to_72d(self.mean)\n        std_72d = subset_271d_to_72d(self.std)\n        return (flow_output - mean_72d) / std_72d\n\ndef sequence_joints_to_features(positions: torch.Tensor, dataset_type: str='t2m', feet_thre: float=0.002) -> torch.Tensor:\n    config = get_dataset_config(dataset_type)\n    raw_offsets = config['raw_offsets']\n    kinematic_chain = config['kinematic_chain']\n    face_joint_indx = config['face_joint_indx']\n    fid_r = config['fid_r']\n    fid_l = config['fid_l']\n    device = positions.device\n    dtype = positions.dtype\n    if positions.ndim == 4:\n        B, N, J, _ = positions.shape\n        features_batch = []\n        for b in range(B):\n            pos_b = positions[b]\n            feat_b = sequence_joints_to_features(pos_b, dataset_type, feet_thre)\n            features_batch.append(feat_b)\n        return torch.stack(features_batch, dim=0)\n    N = positions.shape[0]\n    root_features = torch.zeros(N, 3, device=device, dtype=dtype)\n    root_features[:, 0] = positions[:, 0, 1]\n    if N > 1:\n        root_features[1:, 1] = positions[1:, 0, 0] - positions[:-1, 0, 0]\n        root_features[1:, 2] = positions[1:, 0, 2] - positions[:-1, 0, 2]\n    quaternions = _compute_ik(positions, raw_offsets, kinematic_chain, face_joint_indx)\n    root_quat = quaternions[:, 0].clone()\n    ric = positions - positions[:, 0:1, :]\n    ric = qrot(root_quat.unsqueeze(1).expand(-1, 22, -1), ric)\n    rotations_6d = quaternion_to_cont6d(quaternions)\n    local_vel = torch.zeros(N, 22, 3, device=device, dtype=dtype)\n    if N > 1:\n        local_vel[1:] = qrot(root_quat[1:].unsqueeze(1).expand(-1, 22, -1), positions[1:] - positions[:-1])\n    feet_l = torch.zeros(N, 2, device=device, dtype=dtype)\n    feet_r = torch.zeros(N, 2, device=device, dtype=dtype)\n    if N > 1:\n        vel_l = positions[1:, fid_l] - positions[:-1, fid_l]\n        vel_r = positions[1:, fid_r] - positions[:-1, fid_r]\n        feet_l[1:] = (torch.sum(vel_l ** 2, dim=-1) < feet_thre).float()\n        feet_r[1:] = (torch.sum(vel_r ** 2, dim=-1) < feet_thre).float()\n    features = torch.cat([root_features, ric.reshape(N, -1), rotations_6d.reshape(N, -1), local_vel.reshape(N, -1), feet_l, feet_r], dim=-1)\n    return features\n\ndef features_to_positions(features: torch.Tensor, dataset_type: str='t2m') -> torch.Tensor:\n    root_features = features[..., 0:3]\n    ric = features[..., 3:69].reshape(features.shape[:-1] + (22, 3))\n    rotations_6d = features[..., 69:201].reshape(features.shape[:-1] + (22, 6))\n    root_quat = cont6d_to_quaternion(rotations_6d[..., 0, :])\n    root_height_y = root_features[..., 0:1]\n    root_vel_x = root_features[..., 1:2]\n    root_vel_z = root_features[..., 2:3]\n    if features.ndim == 2:\n        root_pos_x = torch.cumsum(root_vel_x, dim=0)\n        root_pos_z = torch.cumsum(root_vel_z, dim=0)\n    else:\n        root_pos_x = torch.cumsum(root_vel_x, dim=-2)\n        root_pos_z = torch.cumsum(root_vel_z, dim=-2)\n    global_root_pos = torch.cat([root_pos_x, root_height_y, root_pos_z], dim=-1)\n    root_quat_expanded = root_quat.unsqueeze(-2).expand(root_quat.shape[:-1] + (22, -1))\n    positions = global_root_pos.unsqueeze(-2) + qrot(qinv(root_quat_expanded), ric)\n    return positions\n\ndef flow_output_to_positions(flow_output: torch.Tensor, prev_root_pos: torch.Tensor, prev_root_rot_6d: torch.Tensor) -> torch.Tensor:\n    B = flow_output.shape[0]\n    device = flow_output.device\n    dtype = flow_output.dtype\n    root_height = flow_output[:, 0:1]\n    root_vel = flow_output[:, 1:3]\n    root_rot_6d = flow_output[:, 3:9]\n    joint_ric = flow_output[:, 9:72].reshape(B, 21, 3)\n    new_root_x = prev_root_pos[:, 0:1] + root_vel[:, 0:1]\n    new_root_y = root_height\n    new_root_z = prev_root_pos[:, 2:3] + root_vel[:, 1:2]\n    new_root_pos = torch.cat([new_root_x, new_root_y, new_root_z], dim=-1)\n    root_quat = cont6d_to_quaternion(root_rot_6d)\n    root_quat_expanded = root_quat.unsqueeze(1).expand(-1, 21, -1)\n    global_joint_offsets = qrot(qinv(root_quat_expanded), joint_ric)\n    global_joints = new_root_pos.unsqueeze(1) + global_joint_offsets\n    positions = torch.cat([new_root_pos.unsqueeze(1), global_joints], dim=1)\n    return positions\n\ndef flow_output_to_displacements(flow_output: torch.Tensor) -> torch.Tensor:\n    B = flow_output.shape[0]\n    device = flow_output.device\n    dtype = flow_output.dtype\n    root_disp_x = flow_output[:, 1:2]\n    root_disp_z = flow_output[:, 2:3]\n    root_disp_y = torch.zeros_like(root_disp_x)\n    root_disp = torch.cat([root_disp_x, root_disp_y, root_disp_z], dim=-1)\n    joint_disps = flow_output[:, 9:72].reshape(B, 21, 3)\n    displacements = torch.cat([root_disp.unsqueeze(1), joint_disps], dim=1)\n    return displacements\n\ndef extract_prev_frame_features(frame: torch.Tensor) -> torch.Tensor:\n    root_height = frame[:, 0:1]\n    root_vel = frame[:, 1:3]\n    root_rot6d = frame[:, 69:75]\n    root_features = torch.cat([root_height, root_vel, root_rot6d], dim=-1)\n    joint_ric = frame[:, 6:69]\n    joint_rot = frame[:, 75:201]\n    joint_vel = frame[:, 204:267]\n    joint_features = torch.cat([joint_ric, joint_rot, joint_vel], dim=-1)\n    return torch.cat([root_features, joint_features], dim=-1)\n\ndef flow_output_to_271d(flow_output: torch.Tensor, prev_frame: torch.Tensor, prev_root_pos: torch.Tensor, dataset_type: str='t2m', feet_thre: float=0.002):\n    B = flow_output.shape[0]\n    device = flow_output.device\n    dtype = flow_output.dtype\n    config = get_dataset_config(dataset_type)\n    fid_r = config['fid_r']\n    fid_l = config['fid_l']\n    raw_offsets = config['raw_offsets'].to(device=device, dtype=dtype)\n    kinematic_chain = config['kinematic_chain']\n    face_joint_indx = config['face_joint_indx']\n    root_height = flow_output[:, 0:1]\n    root_vel = flow_output[:, 1:3]\n    root_rot_6d = flow_output[:, 3:9]\n    joint_ric_21 = flow_output[:, 9:72].reshape(B, 21, 3)\n    new_root_x = prev_root_pos[:, 0:1] + root_vel[:, 0:1]\n    new_root_y = root_height\n    new_root_z = prev_root_pos[:, 2:3] + root_vel[:, 1:2]\n    new_root_pos = torch.cat([new_root_x, new_root_y, new_root_z], dim=-1)\n    root_ric = torch.zeros(B, 1, 3, device=device, dtype=dtype)\n    ric = torch.cat([root_ric, joint_ric_21], dim=1)\n    prev_root_rot_6d = prev_frame[:, 69:75]\n    prev_root_quat = cont6d_to_quaternion(prev_root_rot_6d)\n    prev_ric = prev_frame[:, 3:69].reshape(B, 22, 3)\n    prev_root_quat_exp = prev_root_quat.unsqueeze(1).expand(-1, 22, -1)\n    prev_positions = prev_root_pos.unsqueeze(1) + qrot(qinv(prev_root_quat_exp), prev_ric)\n    root_quat = cont6d_to_quaternion(root_rot_6d)\n    root_quat_exp = root_quat.unsqueeze(1).expand(-1, 21, -1)\n    global_offsets = qrot(qinv(root_quat_exp), joint_ric_21)\n    global_joints = new_root_pos.unsqueeze(1) + global_offsets\n    new_positions = torch.cat([new_root_pos.unsqueeze(1), global_joints], dim=1)\n    quaternions = _compute_ik(new_positions, raw_offsets, kinematic_chain, face_joint_indx)\n    rotations_6d = quaternion_to_cont6d(quaternions)\n    pos_delta = new_positions - prev_positions\n    root_quat_expanded = root_quat.unsqueeze(1).expand(-1, 22, -1)\n    local_vel = qrot(root_quat_expanded, pos_delta)\n    vel_l = new_positions[:, fid_l] - prev_positions[:, fid_l]\n    vel_r = new_positions[:, fid_r] - prev_positions[:, fid_r]\n    feet_l = (torch.sum(vel_l ** 2, dim=-1) < feet_thre).float()\n    feet_r = (torch.sum(vel_r ** 2, dim=-1) < feet_thre).float()\n    foot_contacts = torch.cat([feet_l, feet_r], dim=-1)\n    root_features = torch.cat([root_height, root_vel], dim=-1)\n    new_frame = torch.cat([root_features, ric.reshape(B, -1), rotations_6d.reshape(B, -1), local_vel.reshape(B, -1), foot_contacts], dim=-1)\n    new_frame = new_frame.to(device=device, dtype=dtype)\n    new_root_pos = new_root_pos.to(device=device, dtype=dtype)\n    return (new_frame, new_root_pos)\n\ndef generated_positions_to_271d(new_positions: torch.Tensor, prev_positions: Optional[torch.Tensor]=None, dataset_type: str='t2m', feet_thre: float=0.002, fk_offsets: Optional[torch.Tensor]=None, normalizer: Optional['FeatureNormalizer']=None, **kwargs) -> Tuple[torch.Tensor, torch.Tensor, Optional[torch.Tensor]]:\n    if new_positions.ndim != 3 or new_positions.shape[-2:] != (22, 3):\n        raise ValueError(f'Expected new_positions shape (B, 22, 3), got {new_positions.shape}')\n    B = new_positions.shape[0]\n    device = new_positions.device\n    dtype = new_positions.dtype\n    cfg = get_dataset_config(dataset_type)\n    raw_offsets = cfg['raw_offsets'].to(device=device, dtype=dtype)\n    kinematic_chain = cfg['kinematic_chain']\n    face_joint_indx = cfg['face_joint_indx']\n    fid_l = cfg['fid_l']\n    fid_r = cfg['fid_r']\n    if prev_positions is not None:\n        if prev_positions.shape != (B, 22, 3):\n            raise ValueError(f'Expected prev_positions shape {(B, 22, 3)}, got {prev_positions.shape}')\n        if prev_positions.device != device or prev_positions.dtype != dtype:\n            prev_positions_resolved = prev_positions.to(device=device, dtype=dtype)\n        else:\n            prev_positions_resolved = prev_positions\n    else:\n        prev_positions_resolved = None\n    quaternions = _compute_ik(new_positions, raw_offsets, kinematic_chain, face_joint_indx)\n    rotations_6d = quaternion_to_cont6d(quaternions)\n    root_quat = quaternions[:, 0]\n    root_quat_expanded = root_quat.unsqueeze(1).expand(-1, 22, -1)\n    new_root_pos = new_positions[:, 0]\n    fk_positions = None\n    if fk_offsets is not None:\n        fk_positions = _forward_kinematics(rotations_6d, new_root_pos, fk_offsets, kinematic_chain)\n        new_positions = fk_positions\n    root_height_y = new_root_pos[:, 1:2]\n    if prev_positions_resolved is None:\n        root_vel_x = torch.zeros((B, 1), device=device, dtype=dtype)\n        root_vel_z = torch.zeros((B, 1), device=device, dtype=dtype)\n    else:\n        root_vel_x = new_root_pos[:, 0:1] - prev_positions_resolved[:, 0, 0:1]\n        root_vel_z = new_root_pos[:, 2:3] - prev_positions_resolved[:, 0, 2:3]\n    root_features = torch.cat([root_height_y, root_vel_x, root_vel_z], dim=-1)\n    ric_source = new_positions\n    ric = ric_source - ric_source[:, 0:1]\n    ric = qrot(root_quat_expanded, ric)\n    if prev_positions_resolved is None:\n        local_vel = torch.zeros((B, 22, 3), device=device, dtype=dtype)\n        feet_l = torch.zeros((B, 2), device=device, dtype=dtype)\n        feet_r = torch.zeros((B, 2), device=device, dtype=dtype)\n    else:\n        pos_delta = new_positions - prev_positions_resolved\n        local_vel = qrot(root_quat_expanded, pos_delta)\n        vel_l = new_positions[:, fid_l] - prev_positions_resolved[:, fid_l]\n        vel_r = new_positions[:, fid_r] - prev_positions_resolved[:, fid_r]\n        feet_l = (torch.sum(vel_l ** 2, dim=-1) < feet_thre).float()\n        feet_r = (torch.sum(vel_r ** 2, dim=-1) < feet_thre).float()\n    foot_contacts = torch.cat([feet_l, feet_r], dim=-1)\n    new_frame = torch.cat([root_features, ric.reshape(B, -1), rotations_6d.reshape(B, -1), local_vel.reshape(B, -1), foot_contacts], dim=-1)\n    if normalizer is not None:\n        new_frame = normalizer.normalize(new_frame)\n    return (new_frame, new_root_pos, fk_positions)\n\nclass RootPositionTracker:\n\n    def __init__(self, initial_root_pos: torch.Tensor):\n        if initial_root_pos.dim() != 2 or initial_root_pos.size(-1) != 3:\n            raise ValueError('initial_root_pos must be shape (B, 3)')\n        self.root_pos = initial_root_pos\n\n    @classmethod\n    def from_history(cls, history_271: torch.Tensor):\n        if history_271.dim() != 3 or history_271.size(-1) < 3:\n            raise ValueError('history_271 must be shape (B, T, 271)')\n        device = history_271.device\n        dtype = history_271.dtype\n        root_height = history_271[..., 0]\n        root_vel_x = history_271[..., 1]\n        root_vel_z = history_271[..., 2]\n        root_pos_x = torch.cumsum(root_vel_x, dim=1)\n        root_pos_z = torch.cumsum(root_vel_z, dim=1)\n        final_x = root_pos_x[:, -1]\n        final_z = root_pos_z[:, -1]\n        final_y = root_height[:, -1]\n        initial_root_pos = torch.stack([final_x, final_y, final_z], dim=-1).to(device=device, dtype=dtype)\n        return cls(initial_root_pos)\n\n    def get(self):\n        return self.root_pos\n\n    def set(self, new_root_pos: torch.Tensor):\n        self.root_pos = new_root_pos\n\n    def update(self, new_frame_271: torch.Tensor):\n        if new_frame_271.device != self.root_pos.device:\n            raise RuntimeError(f'Device mismatch in RootPositionTracker.update(): {new_frame_271.device} vs {self.root_pos.device}')\n        root_height = new_frame_271[:, 0:1]\n        root_vel = new_frame_271[:, 1:3]\n        new_x = self.root_pos[:, 0:1] + root_vel[:, 0:1]\n        new_z = self.root_pos[:, 2:3] + root_vel[:, 1:2]\n        new_y = root_height\n        self.root_pos = torch.cat([new_x, new_y, new_z], dim=-1)",
        'utils/visualization.py': 'import numpy as np\nimport matplotlib.pyplot as plt\nfrom matplotlib.animation import FuncAnimation\nfrom pathlib import Path\nfrom typing import Optional, Any\nfrom utils.motion_utils import T2M_KINEMATIC_CHAIN\n\ndef probe_camera_state(ax) -> dict:\n    elev = ax.elev\n    azim = ax.azim\n    xlim = ax.get_xlim3d()\n    ylim = ax.get_ylim3d()\n    zlim = ax.get_zlim3d()\n    xr = xlim[1] - xlim[0]\n    yr = ylim[1] - ylim[0]\n    zr = zlim[1] - zlim[0]\n    max_range = max(xr, yr, zr)\n    state = dict(elev=elev, azim=azim, xlim=xlim, ylim=ylim, zlim=zlim, x_range=xr, y_range=yr, z_range=zr, plotly_aspectratio=dict(x=xr / max_range, y=yr / max_range, z=zr / max_range), plotly_camera_eye=dict(x=float(1.75 * np.cos(np.deg2rad(elev)) * np.cos(np.deg2rad(azim))), y=float(1.75 * np.cos(np.deg2rad(elev)) * np.sin(np.deg2rad(azim))), z=float(1.75 * np.sin(np.deg2rad(elev)))))\n    print(\'=\' * 50)\n    print(f\'  elev       : {elev:.2f} deg\')\n    print(f\'  azim       : {azim:.2f} deg\')\n    print(f\'  xlim       : [{xlim[0]:.3f}, {xlim[1]:.3f}]  range={xr:.3f}\')\n    print(f\'  ylim       : [{ylim[0]:.3f}, {ylim[1]:.3f}]  range={yr:.3f}\')\n    print(f\'  zlim       : [{zlim[0]:.3f}, {zlim[1]:.3f}]  range={zr:.3f}\')\n    print(f"  plotly aspectratio : {state[\'plotly_aspectratio\']}")\n    print(f"  plotly camera eye  : {state[\'plotly_camera_eye\']}")\n    print(\'=\' * 50)\n    return state\n\ndef plot_3d_motion(motion: np.ndarray, fps: float=20, radius: float=1.0, title: str=\'Motion Visualization\', follow_root: bool=False, probe: bool=False, save_path: Optional[Path]=None):\n    import imageio\n    import io\n    import base64\n    from IPython.display import HTML\n    colors = [\'#2980b9\', \'#c0392b\', \'#27ae60\', \'#f39c12\', \'#8e44ad\']\n    pos_min = motion.min(axis=(0, 1))\n    pos_max = motion.max(axis=(0, 1))\n    x_range = [pos_min[0] - radius, pos_max[0] + radius]\n    y_range = [pos_min[2] - radius, pos_max[2] + radius]\n    z_range = [pos_min[1], pos_max[1] + 0.5]\n    fig = plt.figure(figsize=(6, 6), dpi=120)\n    ax = fig.add_subplot(111, projection=\'3d\')\n    ax.xaxis.pane.fill = False\n    ax.yaxis.pane.fill = False\n    ax.zaxis.pane.fill = False\n    ax.xaxis.pane.set_edgecolor(\'lightgray\')\n    ax.yaxis.pane.set_edgecolor(\'lightgray\')\n    ax.zaxis.pane.set_edgecolor(\'lightgray\')\n    ax.grid(False)\n    ax.view_init(elev=15, azim=65)\n    ax.set_xlim3d(x_range)\n    ax.set_ylim3d(y_range)\n    ax.set_zlim3d(z_range)\n    ax.set_xlabel(\'X (Side)\')\n    ax.set_ylabel(\'Z (Forward)\')\n    ax.set_zlabel(\'Y (Height)\')\n    ax.set_title(title)\n    if probe:\n        print(f"\\n[probe] Matplotlib camera + scene state for \'{title}\':")\n        probe_camera_state(ax)\n    lines = [ax.plot([], [], [], color=colors[i % len(colors)], marker=\'o\', ms=2, lw=2)[0] for i in range(len(T2M_KINEMATIC_CHAIN))]\n    if save_path:\n        save_path.parent.mkdir(parents=True, exist_ok=True)\n    target = save_path if save_path else io.BytesIO()\n    writer = imageio.get_writer(target, format=\'mp4\', fps=fps, codec=\'libx264\', output_params=[\'-preset\', \'ultrafast\', \'-crf\', \'28\'])\n    for frame_idx in range(len(motion)):\n        if follow_root:\n            root = motion[frame_idx, 0, :]\n            ax.set_xlim3d([root[0] - radius, root[0] + radius])\n            ax.set_ylim3d([root[2] - radius, root[2] + radius])\n        for i, c_indices in enumerate(T2M_KINEMATIC_CHAIN):\n            joints = motion[frame_idx, c_indices, :]\n            lines[i].set_data(joints[:, 0], joints[:, 2])\n            lines[i].set_3d_properties(joints[:, 1])\n        fig.canvas.draw()\n        img = np.asarray(fig.canvas.buffer_rgba())[..., :3]\n        writer.append_data(img)\n    writer.close()\n    plt.close(fig)\n    if save_path:\n        print(f\'Saved animation to {save_path}\')\n        return save_path\n    target.seek(0)\n    b64 = base64.b64encode(target.read()).decode()\n    return HTML(f\'<video controls width="600"><source src="data:video/mp4;base64,{b64}"></video>\')\n\ndef plot_3d_motion_plotly(motion: np.ndarray, fps: float=20, radius: float=1.0, title: str=\'Motion Visualization\', follow_root: bool=False):\n    try:\n        import plotly.graph_objects as go\n    except ImportError:\n        raise ImportError(\'Please install plotly: pip install plotly\')\n    n_frames = len(motion)\n    pos_min = motion.min(axis=(0, 1))\n    pos_max = motion.max(axis=(0, 1))\n    if follow_root:\n        root0 = motion[0, 0, :]\n        x_range = [root0[0] - radius, root0[0] + radius]\n        y_range = [root0[2] - radius, root0[2] + radius]\n        z_range = [pos_min[1], pos_max[1] + 0.5]\n    else:\n        x_range = [pos_min[0] - radius, pos_max[0] + radius]\n        y_range = [pos_min[2] - radius, pos_max[2] + radius]\n        z_range = [pos_min[1], pos_max[1] + 0.5]\n    xr = x_range[1] - x_range[0]\n    yr = y_range[1] - y_range[0]\n    zr = z_range[1] - z_range[0]\n    max_range = max(xr, yr, zr)\n    aspectratio = dict(x=xr / max_range, y=yr / max_range, z=zr / max_range)\n    elev = 15.0\n    azim = 65.0\n    dist = 1.75\n    camera_eye = dict(x=dist * np.cos(np.deg2rad(elev)) * np.cos(np.deg2rad(azim)), y=dist * np.cos(np.deg2rad(elev)) * np.sin(np.deg2rad(azim)), z=dist * np.sin(np.deg2rad(elev)))\n    colors = [\'#2980b9\', \'#c0392b\', \'#27ae60\', \'#f39c12\', \'#8e44ad\']\n    initial_data = []\n    for i, c_indices in enumerate(T2M_KINEMATIC_CHAIN):\n        joints = motion[0, c_indices, :]\n        initial_data.append(go.Scatter3d(x=joints[:, 0], y=joints[:, 2], z=joints[:, 1], mode=\'lines+markers\', marker=dict(size=2.5, color=colors[i]), line=dict(width=3, color=colors[i]), name=f\'Chain {i}\', showlegend=False, hoverinfo=\'skip\'))\n    frames = []\n    for frame_idx in range(n_frames):\n        frame_data = []\n        for i, c_indices in enumerate(T2M_KINEMATIC_CHAIN):\n            joints = motion[frame_idx, c_indices, :]\n            frame_data.append(go.Scatter3d(x=joints[:, 0], y=joints[:, 2], z=joints[:, 1]))\n        layout_update = {}\n        if follow_root:\n            root = motion[frame_idx, 0, :]\n            layout_update = dict(scene=dict(xaxis=dict(range=[root[0] - radius, root[0] + radius]), yaxis=dict(range=[root[2] - radius, root[2] + radius])))\n        frames.append(go.Frame(data=frame_data, name=str(frame_idx), layout=layout_update))\n    fig = go.Figure(data=initial_data, frames=frames)\n    fig.update_layout(title=title, width=800, height=800, scene=dict(xaxis=dict(title=\'X (Side)\', range=x_range, autorange=False, showbackground=True, backgroundcolor=\'white\', gridcolor=\'lightgray\', zerolinecolor=\'gray\'), yaxis=dict(title=\'Z (Forward)\', range=y_range, autorange=False, showbackground=True, backgroundcolor=\'white\', gridcolor=\'lightgray\', zerolinecolor=\'gray\'), zaxis=dict(title=\'Y (Height)\', range=z_range, autorange=False, showbackground=True, backgroundcolor=\'white\', gridcolor=\'lightgray\', zerolinecolor=\'gray\'), aspectmode=\'manual\', aspectratio=aspectratio, camera=dict(eye=camera_eye, up=dict(x=0, y=0, z=1), projection=dict(type=\'orthographic\'))), updatemenus=[dict(type=\'buttons\', showactive=False, direction=\'left\', x=0.0, y=0, xanchor=\'left\', yanchor=\'top\', buttons=[dict(label=\'▶/⏸\', method=\'animate\', args=[None, dict(frame=dict(duration=1000 / fps, redraw=True), fromcurrent=True, transition=dict(duration=0, easing=\'linear\'))]), dict(label=\'⟲\', method=\'animate\', args=[[str(0)], dict(frame=dict(duration=0, redraw=True), mode=\'immediate\', transition=dict(duration=0))])])], sliders=[dict(active=0, yanchor=\'top\', xanchor=\'left\', currentvalue=dict(font=dict(size=12), prefix=\'Frame: \', visible=True, xanchor=\'right\'), transition=dict(duration=0, easing=\'linear\'), pad=dict(b=10, t=50), len=0.9, x=0.1, y=0, steps=[dict(args=[[str(k)], dict(frame=dict(duration=0, redraw=True), mode=\'immediate\', transition=dict(duration=0))], label=str(k), method=\'animate\') for k in range(n_frames)])], margin=dict(l=0, r=20, t=40, b=0), plot_bgcolor=\'white\', paper_bgcolor=\'white\')\n    return fig\n\ndef visualize_motion(joint_positions: np.ndarray, title: str=\'Motion Visualization\', save_path: Optional[Path]=None, fps: float=20, skip_frames: int=1, radius: float=1, notebook: bool=True, probe: bool=False, backend: str=\'matplotlib\') -> Any:\n    fps = fps / skip_frames\n    motion_subsampled = joint_positions[::skip_frames]\n    if backend == \'plotly\':\n        try:\n            fig = plot_3d_motion_plotly(motion_subsampled, fps=fps, radius=radius, title=title)\n            if save_path:\n                save_path.parent.mkdir(parents=True, exist_ok=True)\n                html_path = save_path.with_suffix(\'.html\')\n                fig.write_html(str(html_path))\n                print(f\'Saved interactive animation to {html_path}\')\n            if notebook:\n                return fig\n            return fig\n        except ImportError:\n            print(\'Plotly not available, falling back to matplotlib...\')\n            backend = \'matplotlib\'\n    if backend == \'matplotlib\':\n        html = plot_3d_motion(motion_subsampled, radius=radius, fps=fps, title=title, probe=probe)\n        return html\n\ndef compare_motions(generated_joints: np.ndarray, ground_truth_joints: np.ndarray, save_path: Optional[Path]=None, backend: str=\'plotly\') -> None:\n    visualize_motion(generated_joints, title=\'Generated vs Ground Truth\', save_path=save_path, backend=backend)',
        'utils/quaternion.py': "import torch\nimport numpy as np\n_EPS4 = np.finfo(float).eps * 4.0\n_FLOAT_EPS = np.finfo(np.float64).eps\n\ndef qinv(q):\n    assert q.shape[-1] == 4, 'q must be a tensor of shape (*, 4)'\n    q_conj = q.clone()\n    q_conj[..., 1:] = -q_conj[..., 1:]\n    return q_conj\n\ndef qmul(q, r):\n    assert q.shape[-1] == 4\n    assert r.shape[-1] == 4\n    qw, qx, qy, qz = torch.unbind(q, dim=-1)\n    rw, rx, ry, rz = torch.unbind(r, dim=-1)\n    w = rw * qw - rx * qx - ry * qy - rz * qz\n    x = rw * qx + rx * qw - ry * qz + rz * qy\n    y = rw * qy + rx * qz + ry * qw - rz * qx\n    z = rw * qz - rx * qy + ry * qx + rz * qw\n    return torch.stack((w, x, y, z), dim=-1)\n\ndef qrot(q, v):\n    assert q.shape[-1] == 4\n    assert v.shape[-1] == 3\n    assert q.shape[:-1] == v.shape[:-1]\n    original_shape = list(v.shape)\n    q = q.contiguous().view(-1, 4)\n    v = v.contiguous().view(-1, 3)\n    qvec = q[:, 1:]\n    uv = torch.cross(qvec, v, dim=1)\n    uuv = torch.cross(qvec, uv, dim=1)\n    return (v + 2 * (q[:, :1] * uv + uuv)).view(original_shape)\n\ndef quaternion_to_matrix(quaternions):\n    r, i, j, k = torch.unbind(quaternions, -1)\n    two_s = 2.0 / (quaternions * quaternions).sum(-1)\n    o = torch.stack((1 - two_s * (j * j + k * k), two_s * (i * j - k * r), two_s * (i * k + j * r), two_s * (i * j + k * r), 1 - two_s * (i * i + k * k), two_s * (j * k - i * r), two_s * (i * k - j * r), two_s * (j * k + i * r), 1 - two_s * (i * i + j * j)), -1)\n    return o.reshape(quaternions.shape[:-1] + (3, 3))\n\ndef quaternion_to_cont6d(quaternions):\n    r, i, j, k = torch.unbind(quaternions, -1)\n    two_s = 2.0 / (quaternions * quaternions).sum(-1)\n    c0_x = 1 - two_s * (j * j + k * k)\n    c0_y = two_s * (i * j + k * r)\n    c0_z = two_s * (i * k - j * r)\n    c1_x = two_s * (i * j - k * r)\n    c1_y = 1 - two_s * (i * i + k * k)\n    c1_z = two_s * (j * k + i * r)\n    return torch.stack((c0_x, c0_y, c0_z, c1_x, c1_y, c1_z), dim=-1)\n\ndef cont6d_to_matrix(cont6d):\n    assert cont6d.shape[-1] == 6, 'The last dimension must be 6'\n    x_raw = cont6d[..., 0:3]\n    y_raw = cont6d[..., 3:6]\n    eps = 1e-08\n    x_norm = torch.norm(x_raw, dim=-1, keepdim=True).clamp(min=eps)\n    x = x_raw / x_norm\n    z = torch.cross(x, y_raw, dim=-1)\n    z_norm = torch.norm(z, dim=-1, keepdim=True).clamp(min=eps)\n    z = z / z_norm\n    y = torch.cross(z, x, dim=-1)\n    x = x[..., None]\n    y = y[..., None]\n    z = z[..., None]\n    mat = torch.cat([x, y, z], dim=-1)\n    return mat\n\ndef matrix_to_quaternion(rotation_matrix):\n    batch_shape = rotation_matrix.shape[:-2]\n    rotation_matrix = rotation_matrix.reshape(-1, 3, 3)\n    batch_size = rotation_matrix.shape[0]\n    q = torch.zeros(batch_size, 4, device=rotation_matrix.device, dtype=rotation_matrix.dtype)\n    trace = rotation_matrix[:, 0, 0] + rotation_matrix[:, 1, 1] + rotation_matrix[:, 2, 2]\n    mask1 = trace > 0\n    s1 = torch.sqrt(trace[mask1] + 1.0) * 2\n    q[mask1, 0] = 0.25 * s1\n    q[mask1, 1] = (rotation_matrix[mask1, 2, 1] - rotation_matrix[mask1, 1, 2]) / s1\n    q[mask1, 2] = (rotation_matrix[mask1, 0, 2] - rotation_matrix[mask1, 2, 0]) / s1\n    q[mask1, 3] = (rotation_matrix[mask1, 1, 0] - rotation_matrix[mask1, 0, 1]) / s1\n    mask2 = ~mask1 & (rotation_matrix[:, 0, 0] > rotation_matrix[:, 1, 1]) & (rotation_matrix[:, 0, 0] > rotation_matrix[:, 2, 2])\n    s2 = torch.sqrt(1.0 + rotation_matrix[mask2, 0, 0] - rotation_matrix[mask2, 1, 1] - rotation_matrix[mask2, 2, 2]) * 2\n    q[mask2, 0] = (rotation_matrix[mask2, 2, 1] - rotation_matrix[mask2, 1, 2]) / s2\n    q[mask2, 1] = 0.25 * s2\n    q[mask2, 2] = (rotation_matrix[mask2, 0, 1] + rotation_matrix[mask2, 1, 0]) / s2\n    q[mask2, 3] = (rotation_matrix[mask2, 0, 2] + rotation_matrix[mask2, 2, 0]) / s2\n    mask3 = ~mask1 & ~mask2 & (rotation_matrix[:, 1, 1] > rotation_matrix[:, 2, 2])\n    s3 = torch.sqrt(1.0 + rotation_matrix[mask3, 1, 1] - rotation_matrix[mask3, 0, 0] - rotation_matrix[mask3, 2, 2]) * 2\n    q[mask3, 0] = (rotation_matrix[mask3, 0, 2] - rotation_matrix[mask3, 2, 0]) / s3\n    q[mask3, 1] = (rotation_matrix[mask3, 0, 1] + rotation_matrix[mask3, 1, 0]) / s3\n    q[mask3, 2] = 0.25 * s3\n    q[mask3, 3] = (rotation_matrix[mask3, 1, 2] + rotation_matrix[mask3, 2, 1]) / s3\n    mask4 = ~mask1 & ~mask2 & ~mask3\n    s4 = torch.sqrt(1.0 + rotation_matrix[mask4, 2, 2] - rotation_matrix[mask4, 0, 0] - rotation_matrix[mask4, 1, 1]) * 2\n    q[mask4, 0] = (rotation_matrix[mask4, 1, 0] - rotation_matrix[mask4, 0, 1]) / s4\n    q[mask4, 1] = (rotation_matrix[mask4, 0, 2] + rotation_matrix[mask4, 2, 0]) / s4\n    q[mask4, 2] = (rotation_matrix[mask4, 1, 2] + rotation_matrix[mask4, 2, 1]) / s4\n    q[mask4, 3] = 0.25 * s4\n    q = q / (torch.norm(q, dim=-1, keepdim=True) + 1e-10)\n    return q.reshape(batch_shape + (4,))\n\ndef cont6d_to_quaternion(cont6d):\n    mat = cont6d_to_matrix(cont6d)\n    return matrix_to_quaternion(mat)",
        'utils/train_utils.py': 'import copy\nimport os\nimport time\nfrom contextlib import contextmanager\nfrom typing import Optional, Tuple, Dict, Any, Union, TypeVar, Generic\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader\nfrom torch.optim.optimizer import Optimizer\nfrom torch.amp.grad_scaler import GradScaler\nfrom tqdm import tqdm\nfrom config import Config\nfrom models import FlowMatchingPredictor, MotionHistoryEncoder\nfrom utils.dataset import Text2MotionDataset\nfrom utils.text_encoder import CLIPEncoder\nfrom utils.wandb_logger import WandbLogger\nfrom utils.motion_utils import FeatureNormalizer, RootPositionTracker, flow_output_to_271d, generated_positions_to_271d, extract_prev_frame_features, get_fk_offsets\nT = TypeVar(\'T\', bound=nn.Module)\n\nclass EMAModel(Generic[T]):\n\n    def __init__(self, model: T, decay: float=0.999):\n        self.decay = decay\n        self.model: T = copy.deepcopy(model)\n        for p in self.model.parameters():\n            p.requires_grad_(False)\n        self.model.eval()\n\n    def update(self, model: T) -> None:\n        with torch.no_grad():\n            for ema_p, p in zip(self.model.parameters(), model.parameters()):\n                ema_p.data.mul_(self.decay).add_(p.data, alpha=1 - self.decay)\n\n    def to(self, device: str) -> \'EMAModel[T]\':\n        self.model.to(device)\n        return self\n\nclass TimingStats:\n\n    def __init__(self):\n        self.timings: Dict[str, list[float]] = {}\n        self.step_count = 0\n\n    def record(self, key: str, elapsed_ms: float) -> None:\n        if key not in self.timings:\n            self.timings[key] = []\n        self.timings[key].append(elapsed_ms)\n\n    def get_averages(self) -> Dict[str, float]:\n        return {key: sum(times) / len(times) for key, times in self.timings.items() if times}\n\n    def reset(self) -> None:\n        self.timings.clear()\n        self.step_count = 0\n\n    def __str__(self) -> str:\n        averages = self.get_averages()\n        lines = [\'=== Timing Summary ===\']\n        for key in sorted(averages.keys()):\n            lines.append(f\'  {key}: {averages[key]:.2f}ms\')\n        total = sum(averages.values())\n        lines.append(f\'  Total: {total:.2f}ms\')\n        return \'\\n\'.join(lines)\n\n@contextmanager\ndef timer(stats: Optional[TimingStats], key: str):\n    if stats is None:\n        yield\n        return\n    start = time.perf_counter()\n    try:\n        yield\n    finally:\n        elapsed_ms = (time.perf_counter() - start) * 1000\n        stats.record(key, elapsed_ms)\n\nclass Trainer:\n\n    def __init__(self, encoder: MotionHistoryEncoder, predictor: FlowMatchingPredictor, dataloader: DataLoader, config: \'Config\', clip_encoder: Optional[CLIPEncoder]=None, wandb_project: Optional[str]=None, wandb_run_name: Optional[str]=None, resume_from: Optional[str]=None, normalizer: Optional[\'FeatureNormalizer\']=None, val_dataloader: Optional[DataLoader]=None) -> None:\n        self.encoder = encoder\n        self.predictor = predictor\n        self.dataloader = dataloader\n        self.config = config\n        self.clip_encoder = clip_encoder\n        self.wandb_project = wandb_project\n        self.wandb_run_name = wandb_run_name\n        self.resume_from = resume_from\n        self.normalizer = normalizer\n        self.val_dataloader = val_dataloader\n        self.timing_stats: Optional[TimingStats] = TimingStats() if config.enable_profiling else None\n        self.consistency_distribution = torch.distributions.Beta(torch.tensor(50.0, device=self.config.device), torch.tensor(5.0, device=self.config.device))\n\n    @staticmethod\n    def extract_clean_target(frame: torch.Tensor) -> torch.Tensor:\n        return frame[..., 3:69].reshape(*frame.shape[:-1], 22, 3)\n\n    @staticmethod\n    def compute_global_relative_shifts(tracks: torch.Tensor) -> torch.Tensor:\n        root_track = tracks[:, :1, :]\n        return tracks - root_track\n\n    @classmethod\n    def _build_models_from_config(cls, config: Config, normalizer: Optional[FeatureNormalizer]=None) -> Tuple[MotionHistoryEncoder, FlowMatchingPredictor]:\n        encoder = MotionHistoryEncoder(frame_feature_dim=config.encoder_motion_dim, text_embedding_dim=config.encoder_text_dim, text_proj_dim=config.encoder_text_proj_dim, model_dim=config.encoder_hidden_dim, per_joint_out_dim=config.encoder_per_joint_dim, num_layers=config.encoder_num_layers, joint_count=config.encoder_num_joints, text_scale=config.encoder_text_scale, dropout=config.encoder_dropout, normalizer=normalizer)\n        predictor = FlowMatchingPredictor(feature_size=config.get_predictor_feature_size(), config=config.predictor_config, out_channels=config.predictor_config.track_dimensionality, use_relative_shift=True)\n        return (encoder, predictor)\n\n    @classmethod\n    def train(cls, config: Config, train_dataloader: DataLoader, val_dataloader: Optional[DataLoader]=None, normalizer: Optional[FeatureNormalizer]=None, clip_encoder: Optional[CLIPEncoder]=None, wandb_project: Optional[str]=None, wandb_run_name: Optional[str]=None, resume_from: Optional[str]=None, encoder_override: Optional[MotionHistoryEncoder]=None, predictor_override: Optional[FlowMatchingPredictor]=None) -> Tuple[EMAModel[MotionHistoryEncoder], EMAModel[FlowMatchingPredictor]]:\n        encoder, predictor = (encoder_override, predictor_override)\n        if encoder is None or predictor is None:\n            encoder, predictor = cls._build_models_from_config(config, normalizer)\n        trainer = cls(encoder=encoder, predictor=predictor, dataloader=train_dataloader, config=config, clip_encoder=clip_encoder, wandb_project=wandb_project, wandb_run_name=wandb_run_name, resume_from=resume_from, normalizer=normalizer if normalizer is not None else encoder.normalizer, val_dataloader=val_dataloader)\n        return trainer._run_training()\n\n    def setup_training_environment(self) -> Tuple[Any, Any, Optional[WandbLogger], EMAModel[MotionHistoryEncoder], EMAModel[FlowMatchingPredictor], Optimizer, GradScaler, int, Dict[str, Any], str, bool]:\n        device = self.config.device\n        lr = self.config.learning_rate\n        weight_decay = self.config.weight_decay\n        ema_decay = self.config.ema_decay\n        horizon = self.config.horizon\n        curriculum = self.config.curriculum\n        cfg_dropout = self.config.cfg_dropout\n        num_epochs = self.config.num_epochs\n        os.makedirs(self.config.checkpoint_dir, exist_ok=True)\n        self.encoder.to(device)\n        self.predictor.to(device)\n        wandb_logger = None\n        if self.wandb_project:\n            wandb_config = {\'lr\': lr, \'weight_decay\': weight_decay, \'ema_decay\': ema_decay, \'num_epochs\': num_epochs, \'horizon\': horizon, \'cfg_dropout\': cfg_dropout, \'batch_size\': self.dataloader.batch_size, \'encoder_params\': sum((p.numel() for p in self.encoder.parameters())), \'predictor_params\': sum((p.numel() for p in self.predictor.parameters())), \'curriculum\': curriculum}\n            wandb_logger = WandbLogger(project=self.wandb_project, name=self.wandb_run_name, config=wandb_config)\n        encoder_ema = EMAModel(self.encoder, decay=ema_decay).to(device)\n        predictor_ema = EMAModel(self.predictor, decay=ema_decay).to(device)\n        params = list(self.encoder.parameters()) + list(self.predictor.parameters())\n        optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)\n        device_str = str(device)\n        use_amp = device_str.startswith(\'cuda\')\n        scaler = GradScaler(\'cuda\', enabled=use_amp)\n        training_state: Dict[str, Any] = {\'global_step\': 0, \'best_loss\': float(\'inf\'), \'best_epoch\': -1, \'best_val_loss\': float(\'inf\'), \'best_val_epoch\': -1}\n        start_epoch = 0\n        if self.resume_from is not None and os.path.exists(self.resume_from):\n            print(f\'Resuming from checkpoint: {self.resume_from}\')\n            checkpoint = torch.load(self.resume_from, map_location=device, weights_only=False)\n            self.encoder.load_state_dict(checkpoint[\'encoder\'])\n            self.predictor.load_state_dict(checkpoint[\'predictor\'])\n            encoder_ema.model.load_state_dict(checkpoint[\'encoder_ema\'])\n            predictor_ema.model.load_state_dict(checkpoint[\'predictor_ema\'])\n            optimizer.load_state_dict(checkpoint[\'optimizer\'])\n            scaler.load_state_dict(checkpoint[\'scaler\'])\n            start_epoch = checkpoint.get(\'epoch\', 0) + 1\n            training_state[\'global_step\'] = checkpoint.get(\'global_step\', 0)\n            training_state[\'best_loss\'] = checkpoint.get(\'best_loss\', float(\'inf\'))\n            training_state[\'best_epoch\'] = checkpoint.get(\'best_epoch\', -1)\n            training_state[\'best_val_loss\'] = checkpoint.get(\'best_val_loss\', float(\'inf\'))\n            training_state[\'best_val_epoch\'] = checkpoint.get(\'best_val_epoch\', -1)\n            if \'current_horizon\' in checkpoint:\n                training_state[\'current_horizon\'] = checkpoint[\'current_horizon\']\n            print(f"Resumed from epoch {start_epoch}, step {training_state[\'global_step\']}")\n        if curriculum is not None and len(curriculum) > 0:\n            print(f\'Training for {num_epochs} epochs with curriculum: {curriculum}\')\n        else:\n            print(f\'Training for {num_epochs} epochs with fixed horizon={horizon}\')\n        print(f\'Encoder params: {sum((p.numel() for p in self.encoder.parameters())):,}\')\n        print(f\'Predictor params: {sum((p.numel() for p in self.predictor.parameters())):,}\')\n        self.encoder.train()\n        self.predictor.train()\n        return (device, str(self.config.checkpoint_dir), wandb_logger, encoder_ema, predictor_ema, optimizer, scaler, start_epoch, training_state, device_str, use_amp)\n\n    def setup_curriculum_state(self, checkpoint_state: Optional[dict]=None) -> dict:\n        curriculum = self.config.curriculum\n        use_curriculum = curriculum is not None and len(curriculum) > 0\n        if checkpoint_state and \'current_horizon\' in checkpoint_state:\n            current_horizon = checkpoint_state[\'current_horizon\']\n            max_horizon = checkpoint_state[\'max_horizon\']\n        elif use_curriculum and curriculum:\n            current_horizon = curriculum[0][\'horizon\']\n            max_horizon = curriculum[-1][\'horizon\']\n        else:\n            current_horizon = self.config.horizon\n            max_horizon = self.config.horizon\n        return {\'use_curriculum\': use_curriculum, \'current_horizon\': current_horizon, \'max_horizon\': max_horizon}\n\n    def save_training_checkpoint(self, save_dir: str, filename: str, encoder_ema: EMAModel[MotionHistoryEncoder], predictor_ema: EMAModel[FlowMatchingPredictor], optimizer: Optimizer, scaler: GradScaler, epoch: int, global_step: int, loss: float, curriculum_state: dict, training_state: dict) -> None:\n        path = os.path.join(save_dir, filename)\n        checkpoint = {\'encoder\': self.encoder.state_dict(), \'predictor\': self.predictor.state_dict(), \'encoder_ema\': encoder_ema.model.state_dict(), \'predictor_ema\': predictor_ema.model.state_dict(), \'optimizer\': optimizer.state_dict(), \'scaler\': scaler.state_dict(), \'epoch\': epoch, \'global_step\': global_step, \'loss\': loss, \'horizon\': curriculum_state[\'max_horizon\'], \'current_horizon\': curriculum_state[\'current_horizon\'], \'use_curriculum\': curriculum_state[\'use_curriculum\'], \'best_loss\': training_state[\'best_loss\'], \'best_epoch\': training_state[\'best_epoch\'], \'best_val_loss\': training_state[\'best_val_loss\'], \'best_val_epoch\': training_state[\'best_val_epoch\']}\n        checkpoint[\'config\'] = self.config\n        torch.save(checkpoint, path)\n        print(f\'Saved checkpoint: {path}\')\n\n    def unpack_batch(self, batch: dict, device: Union[str, torch.device]) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, int, int]:\n        motion_raw = batch[\'motion\'].to(device)\n        joints = batch[\'joints\'].to(device)\n        B, T, _ = motion_raw.shape\n        lengths = batch.get(\'lengths\', torch.full((B,), T, device=device, dtype=torch.long))\n        motion = self.normalizer.normalize(motion_raw) if self.normalizer is not None else motion_raw\n        if joints.shape[0] != B or joints.shape[1] != T:\n            raise ValueError(f\'Batch shape mismatch between motion {motion.shape} and joints {joints.shape}.\')\n        if \'text_clip\' in batch:\n            text = batch[\'text_clip\'].to(device)\n        elif \'captions\' in batch and self.clip_encoder is not None:\n            captions = batch[\'captions\']\n            with torch.no_grad():\n                text = self.clip_encoder(captions)\n        elif \'captions\' in batch:\n            raise ValueError("Raw captions provided but no clip_encoder. Pass clip_encoder to Trainer() or provide pre-encoded \'text_clip\'.")\n        else:\n            raise ValueError("No text input found. Batch must contain \'text_clip\' or \'captions\'.")\n        if text.ndim != 3 or text.shape[0] != B or text.shape[1] != 1:\n            raise ValueError(f\'Invalid text embedding shape {tuple(text.shape)}. Expected (B, 1, D) with B={B}.\')\n        return (motion, joints, text, lengths, B, T)\n\n    def sample_next_frame_window(self, motion: torch.Tensor, joints: torch.Tensor, curr_horizon: int):\n        _, T, _ = motion.shape\n        assert curr_horizon <= T - 1, f\'curr_horizon {curr_horizon} > T-1 {T - 1}\'\n        if joints.shape[:2] != motion.shape[:2]:\n            raise ValueError(f\'sample_next_frame_window got motion shape {motion.shape} and joints shape {joints.shape}.\')\n        relative_shifts = joints[:, 1:] - joints[:, :-1]\n        return (motion[:, 1:], joints[:, 1:], relative_shifts, curr_horizon)\n\n    def get_rollout_probability(self, epoch: int, total_epochs: int) -> float:\n        if total_epochs <= 1:\n            return self.config.rollout_prob_end\n        progress = float(epoch) / float(total_epochs - 1)\n        progress = max(0.0, min(1.0, progress))\n        return self.config.rollout_prob_start + (self.config.rollout_prob_end - self.config.rollout_prob_start) * progress\n\n    def get_rollout_mask(self, batch_size: int, rollout_prob: float, device: Union[str, torch.device], stochastic: bool) -> torch.Tensor:\n        p = float(max(0.0, min(1.0, rollout_prob)))\n        if p == 0.0:\n            return torch.zeros(batch_size, dtype=torch.bool, device=device)\n        if p == 1.0:\n            return torch.ones(batch_size, dtype=torch.bool, device=device)\n        if stochastic:\n            return torch.rand(batch_size, device=device) < p\n        threshold = max(1, int(round(p * batch_size)))\n        return torch.arange(batch_size, device=device) < threshold\n\n    def incremental_flow_loss(self, motion: torch.Tensor, joints: torch.Tensor, relative_shifts: torch.Tensor, text_for_encoder: torch.Tensor, epoch: int, total_epochs: int, device: Union[str, torch.device], stochastic_rollout: bool, encoder: Optional[MotionHistoryEncoder]=None, predictor: Optional[FlowMatchingPredictor]=None) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, int, float]:\n        enc = encoder if encoder is not None else self.encoder\n        pred_model = predictor if predictor is not None else self.predictor\n        if not hasattr(enc, \'gru_step\'):\n            raise AttributeError(\'Encoder must expose a gru_step(x_t, text_emb, h) method.\')\n        B, _, _ = motion.shape\n        _, j_len, _, _ = joints.shape\n        if j_len <= 1:\n            raise ValueError(\'No prediction steps found in target window.\')\n        if motion.shape[1] != j_len:\n            raise ValueError(f\'motion length {motion.shape[1]} != joints length {j_len}.\')\n        fk_offsets = get_fk_offsets(joints) if self.config.use_fk else None\n        pred_steps = j_len - 1\n        hist = motion[:, :-1].clone()\n        hist_joints = joints[:, :-1].clone()\n        prev_relative_shifts = relative_shifts[:, :-1].clone()\n        target_motion = motion[:, 1:].clone()\n        target_joints = joints[:, 1:].clone()\n        target_relative_shifts = relative_shifts[:, 1:].clone()\n        with timer(self.timing_stats, \'forward/gru_init\'):\n            context, h_state = enc.gru_step(hist[:, 0], text_for_encoder, h=None)\n        if context is None:\n            raise RuntimeError(\'Encoder context was not initialized from history.\')\n        flow_losses: list[torch.Tensor] = []\n        consistency_losses: list[torch.Tensor] = []\n        rollout_prob = self.get_rollout_probability(epoch=epoch, total_epochs=total_epochs)\n        ode_steps = max(1, int(self.config.rollout_integration_steps))\n        dt = 1.0 / float(ode_steps)\n        contexts = [context]\n        for step_idx in range(pred_steps - 1):\n            current_positions = hist_joints[:, step_idx]\n            rollout_mask = self.get_rollout_mask(B, rollout_prob, device, stochastic_rollout)\n            next_motion = target_motion[:, step_idx].clone()\n            next_joints = target_joints[:, step_idx].clone()\n            if rollout_mask.any():\n                predictor_was_training = pred_model.training\n                pred_model.eval()\n                rolled_count = int(rollout_mask.sum().item())\n                context_roll = context[rollout_mask]\n                text_roll = text_for_encoder[rollout_mask]\n                current_positions_roll = current_positions[rollout_mask]\n                prev_relative_shifts_roll = prev_relative_shifts[rollout_mask, step_idx]\n                x1 = target_relative_shifts[:, step_idx]\n                x_t_roll = torch.randn_like(x1[rollout_mask])\n                pred_roll_endpoint: Optional[torch.Tensor] = None\n                try:\n                    with timer(self.timing_stats, \'forward/rollout_ode\'):\n                        with torch.no_grad():\n                            for ode_step in range(ode_steps):\n                                with timer(self.timing_stats, \'forward/rollout_ode_step\'):\n                                    tau = torch.full((rolled_count,), float(ode_step) * dt, device=device)\n                                    pred_roll = pred_model.forward(track_features=context_roll, noised_tracks=x_t_roll, timesteps=tau, prev_relative_shifts=prev_relative_shifts_roll, text_embedding=text_roll, output_attentions=False, output_hidden_states=False)[0]\n                                    x_t_roll = x_t_roll + pred_roll * dt\n                finally:\n                    if predictor_was_training:\n                        pred_model.train()\n                with timer(self.timing_stats, \'forward/pos_transform\'):\n                    pred_positions_roll = current_positions_roll + x_t_roll\n                    rollout_frame, _, fk_positions_roll = generated_positions_to_271d(new_positions=pred_positions_roll, prev_positions=current_positions_roll, dataset_type=\'t2m\', fk_offsets=fk_offsets[rollout_mask] if fk_offsets is not None else None, normalizer=self.normalizer)\n                next_motion[rollout_mask] = rollout_frame\n                next_joints[rollout_mask] = fk_positions_roll if fk_positions_roll is not None else pred_positions_roll\n                hist[:, step_idx + 1] = next_motion.detach()\n                hist_joints[:, step_idx + 1] = next_joints.detach()\n                prev_relative_shifts[:, step_idx + 1] = (next_joints - current_positions).detach()\n            with timer(self.timing_stats, \'forward/gru_step\'):\n                context, h_state = enc.gru_step(hist[:, step_idx + 1], text_for_encoder, h_state)\n            contexts.append(context)\n        contexts_stacked = torch.stack(contexts, dim=1)\n        contexts_reshaped = contexts_stacked.flatten(0, 1)\n        prev_relative_shifts_reshaped = prev_relative_shifts.flatten(0, 1)\n        text_batched = text_for_encoder.unsqueeze(1).expand(-1, pred_steps, -1).reshape(B * pred_steps, -1)\n        x1 = target_relative_shifts.flatten(0, 1)\n        x0 = torch.randn_like(x1)\n        with timer(self.timing_stats, \'forward/predictor\'):\n            t = torch.rand((x1.shape[0],), device=device)\n            _t = t[:, None, None]\n            xt = _t * x1 + (1 - _t) * x0\n            pred, _, _ = pred_model.forward(track_features=contexts_reshaped, noised_tracks=xt, timesteps=t, prev_relative_shifts=prev_relative_shifts_reshaped, text_embedding=text_batched)\n            flow_loss = F.mse_loss(pred, x1 - x0)\n        consistency_loss = torch.tensor(0.0, device=device)\n        if self.config.use_consistency_loss:\n            hist_joints_reshaped = hist_joints.flatten(0, 1)\n            with timer(self.timing_stats, \'forward/consistency_loss\'):\n                t = self.consistency_distribution.sample((x1.shape[0],))\n                _t = t[:, None, None]\n                xt = _t * x1 + (1 - _t) * x0\n                pred_t, _, _ = pred_model.forward(track_features=contexts_reshaped, noised_tracks=xt, timesteps=t, prev_relative_shifts=prev_relative_shifts_reshaped, text_embedding=text_batched)\n                dt = 1 - _t\n                pred_x1 = xt + pred_t * dt\n            with timer(self.timing_stats, \'forward/pos_transform\'):\n                pred_positions = hist_joints_reshaped + pred_x1\n                pred_frames, _, fk_positions = generated_positions_to_271d(new_positions=pred_positions, prev_positions=hist_joints_reshaped, dataset_type=\'t2m\', fk_offsets=torch.repeat_interleave(fk_offsets, pred_steps, dim=0) if fk_offsets is not None else None, normalizer=self.normalizer)\n                pred_shift = fk_positions - hist_joints_reshaped if fk_positions is not None else pred_x1\n                consistency_loss = F.mse_loss(pred_shift, x1)\n        total_loss = flow_loss + self.config.consistency_loss_weight * consistency_loss\n        return (total_loss, flow_loss, consistency_loss, pred_steps, rollout_prob)\n\n    def apply_cfg_dropout(self, text: torch.Tensor, device: Union[str, torch.device], batch_size: int) -> Optional[torch.Tensor]:\n        if self.config.cfg_dropout <= 0.0:\n            return text\n        if torch.rand(1).item() > self.config.cfg_dropout:\n            return text\n        return None\n\n    def prepare_text_for_encoder(self, text_input: Optional[torch.Tensor], device: Union[str, torch.device], batch_size: int) -> torch.Tensor:\n        if text_input is None:\n            return torch.zeros(batch_size, self.config.encoder_text_dim, device=device)\n        if text_input.ndim != 3:\n            raise ValueError(f\'Invalid text input rank {text_input.ndim}. Expected rank 3 with shape (B, 1, {self.config.encoder_text_dim}).\')\n        if text_input.shape[0] != batch_size:\n            raise ValueError(f\'Invalid text batch size {text_input.shape[0]}. Expected {batch_size}.\')\n        if text_input.shape[1] != 1 or text_input.shape[2] != self.config.encoder_text_dim:\n            raise ValueError(f\'Invalid text input shape {tuple(text_input.shape)}. Expected (B, 1, {self.config.encoder_text_dim}).\')\n        return text_input[:, 0, :]\n\n    def log_batch_metrics(self, wandb_logger: Optional[WandbLogger], loss: torch.Tensor, lr: float, epoch: int, grad_norm: torch.Tensor, batch_time: float, B: int, current_horizon: int, effective_horizon: int, num_pred_frames: int, loss_components: dict, global_step: int) -> None:\n        if wandb_logger is None:\n            return\n        log_dict = {}\n        for key, value in loss_components.items():\n            log_dict[f\'train/loss_{key}\'] = value.item() if hasattr(value, \'item\') else value\n        log_dict.update({\'train/loss\': loss.item(), \'train/lr\': lr, \'train/epoch\': epoch, \'train/grad_norm\': grad_norm.item() if hasattr(grad_norm, \'item\') else grad_norm, \'train/batch_time\': batch_time, \'train/samples_per_sec\': B / batch_time if batch_time > 0 else 0, \'train/current_horizon\': current_horizon, \'train/effective_horizon\': effective_horizon, \'train/num_pred_frames\': num_pred_frames})\n        wandb_logger.log(log_dict, step=global_step)\n\n    def log_epoch_metrics(self, wandb_logger: Optional[WandbLogger], avg_epoch_loss: float, epoch: int, current_horizon: int, val_metrics: dict, global_step: int) -> None:\n        if wandb_logger is None:\n            return\n        log_dict = {\'epoch/avg_loss\': avg_epoch_loss, \'epoch/num\': epoch, \'epoch/current_horizon\': current_horizon}\n        if val_metrics and \'val_loss\' in val_metrics:\n            log_dict[\'epoch/val_loss\'] = val_metrics[\'val_loss\']\n        wandb_logger.log(log_dict, step=global_step)\n\n    def handle_checkpointing(self, save_dir: str, encoder_ema: EMAModel[MotionHistoryEncoder], predictor_ema: EMAModel[FlowMatchingPredictor], optimizer: Optimizer, scaler: GradScaler, epoch: int, global_step: int, avg_epoch_loss: float, curriculum_state: dict, training_state: dict, val_metrics: dict) -> dict:\n        self.save_training_checkpoint(save_dir=save_dir, filename=\'latest.pt\', encoder_ema=encoder_ema, predictor_ema=predictor_ema, optimizer=optimizer, scaler=scaler, epoch=epoch, global_step=global_step, loss=avg_epoch_loss, curriculum_state=curriculum_state, training_state=training_state)\n        if avg_epoch_loss < training_state[\'best_loss\']:\n            tqdm.write(f"New best model! (Loss: {training_state[\'best_loss\']:.6f} -> {avg_epoch_loss:.6f})")\n            training_state[\'best_loss\'] = avg_epoch_loss\n            training_state[\'best_epoch\'] = epoch\n            self.save_training_checkpoint(save_dir=save_dir, filename=\'best.pt\', encoder_ema=encoder_ema, predictor_ema=predictor_ema, optimizer=optimizer, scaler=scaler, epoch=epoch, global_step=global_step, loss=avg_epoch_loss, curriculum_state=curriculum_state, training_state=training_state)\n        if self.config.save_best_val and val_metrics and (\'val_loss\' in val_metrics):\n            val_loss = val_metrics[\'val_loss\']\n            if val_loss < training_state[\'best_val_loss\']:\n                tqdm.write(f"New best validation model! (Val Loss: {training_state[\'best_val_loss\']:.6f} -> {val_loss:.6f})")\n                training_state[\'best_val_loss\'] = val_loss\n                training_state[\'best_val_epoch\'] = epoch\n                self.save_training_checkpoint(save_dir=save_dir, filename=\'best_val.pt\', encoder_ema=encoder_ema, predictor_ema=predictor_ema, optimizer=optimizer, scaler=scaler, epoch=epoch, global_step=global_step, loss=avg_epoch_loss, curriculum_state=curriculum_state, training_state=training_state)\n        return training_state\n\n    def validate(self, encoder: Optional[MotionHistoryEncoder]=None, predictor: Optional[FlowMatchingPredictor]=None, epoch: int=0, total_epochs: Optional[int]=None, horizon: Optional[int]=None, num_batches: Optional[int]=None, device: Optional[Union[str, torch.device]]=None) -> dict:\n        if self.val_dataloader is None:\n            return {}\n        if total_epochs is None:\n            total_epochs = self.config.num_epochs\n        if horizon is None:\n            horizon = self.config.horizon\n        if num_batches is None:\n            num_batches = self.config.val_batches\n        device_value: Union[str, torch.device] = self.config.device if device is None else device\n        enc = encoder if encoder is not None else self.encoder\n        pred = predictor if predictor is not None else self.predictor\n        enc.eval()\n        pred.eval()\n        total_total_loss = 0.0\n        total_flow_loss = 0.0\n        total_consistency_loss = 0.0\n        total_samples = 0\n        pred_horizon = 1\n        self.val_dataloader.dataset.set_horizon(1 + horizon + pred_horizon)\n        with torch.no_grad():\n            for i, batch in enumerate(self.val_dataloader):\n                if num_batches > 0 and i >= num_batches:\n                    break\n                motion, joints, text, _lengths, B, _T = self.unpack_batch(batch=batch, device=device_value)\n                motion, joints, relative_shifts, _ = self.sample_next_frame_window(motion=motion, joints=joints, curr_horizon=horizon)\n                text_for_encoder = self.prepare_text_for_encoder(text, device_value, B)\n                total_loss, flow_loss, consistency_loss, _, _ = self.incremental_flow_loss(motion=motion, joints=joints, relative_shifts=relative_shifts, text_for_encoder=text_for_encoder, epoch=epoch, total_epochs=total_epochs, device=device_value, stochastic_rollout=True, encoder=enc, predictor=pred)\n                total_total_loss += total_loss.item() * B\n                total_flow_loss += flow_loss.item() * B\n                total_consistency_loss += consistency_loss.item() * B\n                total_samples += B\n        enc.train()\n        pred.train()\n        return {\'val_loss\': total_total_loss / max(1, total_samples), \'val_flow_loss\': total_flow_loss / max(1, total_samples), \'val_consistency_loss\': total_consistency_loss / max(1, total_samples)}\n\n    def _run_training(self) -> Tuple[EMAModel[MotionHistoryEncoder], EMAModel[FlowMatchingPredictor]]:\n        device, checkpoint_dir, wandb_logger, encoder_ema, predictor_ema, optimizer, scaler, start_epoch, training_state, device_str, use_amp = self.setup_training_environment()\n        curriculum_state = self.setup_curriculum_state(checkpoint_state=training_state if \'current_horizon\' in training_state else None)\n        num_epochs = self.config.num_epochs\n        lr = self.config.learning_rate\n        max_grad_norm = self.config.gradient_clip\n        val_interval = self.config.val_interval\n        val_batches = self.config.val_batches\n        val_use_ema = self.config.val_use_ema\n        amp_dtype = torch.float32\n        if use_amp:\n            amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n        epoch = start_epoch - 1\n        try:\n            training_state = training_state\n            for epoch in tqdm(range(start_epoch, num_epochs), desc=\'Training\', unit=\'epoch\'):\n                prev_horizon = curriculum_state[\'current_horizon\']\n                if curriculum_state[\'use_curriculum\'] and self.config.curriculum:\n                    for level in reversed(self.config.curriculum):\n                        if epoch <= level[\'epochs\']:\n                            curriculum_state[\'current_horizon\'] = level[\'horizon\']\n                        else:\n                            break\n                    if curriculum_state[\'current_horizon\'] != prev_horizon:\n                        tqdm.write(f"Curriculum update: horizon {prev_horizon} -> {curriculum_state[\'current_horizon\']}")\n                epoch_loss = 0.0\n                num_batches = 0\n                pred_horizon = 1\n                self.dataloader.dataset.set_horizon(1 + curriculum_state[\'current_horizon\'] + pred_horizon)\n                pbar = tqdm(self.dataloader, desc=f\'Epoch {epoch}\', leave=False, unit=\'batch\')\n                batch_start_time = time.time()\n                timing_log_interval = max(1, int(self.config.timing_log_interval))\n                for batch in pbar:\n                    with timer(self.timing_stats, \'data_load\'):\n                        motion, joints, text, lengths, B, T = self.unpack_batch(batch=batch, device=device)\n                        motion, joints, relative_shifts, effective_horizon = self.sample_next_frame_window(motion=motion, joints=joints, curr_horizon=curriculum_state[\'current_horizon\'])\n                    text_input = self.apply_cfg_dropout(text, device, B)\n                    with timer(self.timing_stats, \'text_prep\'):\n                        text_for_encoder = self.prepare_text_for_encoder(text_input, device, B)\n                    optimizer.zero_grad(set_to_none=True)\n                    with timer(self.timing_stats, \'forward\'):\n                        with torch.amp.autocast(device_str, dtype=amp_dtype, enabled=use_amp):\n                            loss, flow_loss, consistency_loss, pred_horizon, rollout_prob = self.incremental_flow_loss(motion=motion, joints=joints, relative_shifts=relative_shifts, text_for_encoder=text_for_encoder, epoch=epoch, total_epochs=num_epochs, device=device, stochastic_rollout=True)\n                    with timer(self.timing_stats, \'backward\'):\n                        scaler.scale(loss).backward()\n                        scaler.unscale_(optimizer)\n                        grad_norm = torch.nn.utils.clip_grad_norm_(list(self.encoder.parameters()) + list(self.predictor.parameters()), max_grad_norm)\n                        scaler.step(optimizer)\n                        scaler.update()\n                    with timer(self.timing_stats, \'ema_update\'):\n                        encoder_ema.update(self.encoder)\n                        predictor_ema.update(self.predictor)\n                    batch_time = time.time() - batch_start_time\n                    batch_start_time = time.time()\n                    pbar.set_postfix({\'loss\': f\'{loss.item():.4f}\', \'L_flow\': f\'{flow_loss.item():.4f}\', \'L_cons\': f\'{consistency_loss.item():.4f}\', \'p_roll\': f\'{rollout_prob:.2f}\', \'lr\': f\'{lr:.2e}\'})\n                    self.log_batch_metrics(wandb_logger=wandb_logger, loss=loss, lr=lr, epoch=epoch, grad_norm=grad_norm, batch_time=batch_time, B=B, current_horizon=curriculum_state[\'current_horizon\'], effective_horizon=effective_horizon, num_pred_frames=pred_horizon, loss_components={\'L_total\': loss.item(), \'L_flow\': flow_loss.item(), \'L_consistency\': consistency_loss.item(), \'rollout_prob\': rollout_prob}, global_step=training_state[\'global_step\'])\n                    if self.timing_stats is not None and (training_state[\'global_step\'] + 1) % timing_log_interval == 0:\n                        timing_dict = self.timing_stats.get_averages()\n                        if wandb_logger is not None:\n                            wandb_logger.log({f\'time/{key}_ms\': val for key, val in timing_dict.items()}, step=training_state[\'global_step\'])\n                        tqdm.write(f"[Step {training_state[\'global_step\']}] {str(self.timing_stats)}")\n                    if training_state[\'global_step\'] % 100 == 0:\n                        tqdm.write(f"[Epoch {epoch}] [Step {training_state[\'global_step\']}] loss={loss.item():.6f} lr={lr:.2e}")\n                    epoch_loss += loss.item()\n                    num_batches += 1\n                    training_state[\'global_step\'] += 1\n                pbar.close()\n                avg_epoch_loss = epoch_loss / max(1, num_batches)\n                tqdm.write(f\'==> End of Epoch {epoch}: Avg Loss = {avg_epoch_loss:.6f}\')\n                val_metrics: dict = {}\n                if self.val_dataloader is not None and (epoch + 1) % val_interval == 0:\n                    tqdm.write(\'Running validation...\')\n                    if val_use_ema:\n                        val_encoder: MotionHistoryEncoder = encoder_ema.model\n                        val_predictor: FlowMatchingPredictor = predictor_ema.model\n                    else:\n                        val_encoder = self.encoder\n                        val_predictor = self.predictor\n                    try:\n                        with timer(self.timing_stats, \'validation\'):\n                            val_metrics = self.validate(encoder=val_encoder, predictor=val_predictor, horizon=curriculum_state[\'current_horizon\'], num_batches=val_batches, device=device, epoch=epoch, total_epochs=num_epochs)\n                        val_loss = val_metrics[\'val_loss\']\n                        tqdm.write(f\'Validation loss: {val_loss:.6f}\')\n                        if wandb_logger is not None:\n                            wandb_logger.log({\'val/loss\': val_loss, \'val/epoch\': epoch}, step=training_state[\'global_step\'])\n                    except Exception as e:\n                        tqdm.write(f\'Validation error: {str(e)}\')\n                        val_metrics = {}\n                self.log_epoch_metrics(wandb_logger=wandb_logger, avg_epoch_loss=avg_epoch_loss, epoch=epoch, current_horizon=curriculum_state[\'current_horizon\'], val_metrics=val_metrics, global_step=training_state[\'global_step\'])\n                if self.config.checkpoint_interval > 0 and (epoch + 1) % self.config.checkpoint_interval == 0 or (val_metrics and val_metrics.get(\'val_loss\', float(\'inf\')) < training_state[\'best_val_loss\']):\n                    with timer(self.timing_stats, \'checkpoint\'):\n                        training_state = self.handle_checkpointing(save_dir=checkpoint_dir, encoder_ema=encoder_ema, predictor_ema=predictor_ema, optimizer=optimizer, scaler=scaler, epoch=epoch, global_step=training_state[\'global_step\'], avg_epoch_loss=avg_epoch_loss, curriculum_state=curriculum_state, training_state=training_state, val_metrics=val_metrics)\n                if self.timing_stats is not None:\n                    tqdm.write(str(self.timing_stats))\n                    if wandb_logger is not None:\n                        epoch_timing = self.timing_stats.get_averages()\n                        wandb_logger.log({f\'epoch_time/{key}_ms\': val for key, val in epoch_timing.items()}, step=training_state[\'global_step\'])\n                    self.timing_stats.reset()\n        except KeyboardInterrupt:\n            tqdm.write(\'Training interrupted. Saving emergency checkpoint...\')\n            self.save_training_checkpoint(save_dir=checkpoint_dir, filename=\'latest_interrupted.pt\', encoder_ema=encoder_ema, predictor_ema=predictor_ema, optimizer=optimizer, scaler=scaler, epoch=epoch, global_step=training_state[\'global_step\'], loss=0.0, curriculum_state=curriculum_state, training_state=training_state)\n            tqdm.write(\'Done.\')\n        if wandb_logger is not None:\n            summary = {\'best_loss\': training_state[\'best_loss\'], \'best_epoch\': training_state[\'best_epoch\'], \'best_val_loss\': training_state[\'best_val_loss\'], \'best_val_epoch\': training_state[\'best_val_epoch\']}\n            if curriculum_state[\'use_curriculum\']:\n                summary[\'final_horizon\'] = curriculum_state[\'current_horizon\']\n                summary[\'max_horizon\'] = curriculum_state[\'max_horizon\']\n            wandb_logger.log_summary(summary)\n            wandb_logger.finish()\n        return (encoder_ema, predictor_ema)',
        'utils/text_encoder.py': '"""\nText encoding utility using CLIP model from Hugging Face Transformers.\n"""\n\nimport torch\nfrom transformers import CLIPTokenizer, CLIPTextModel\nfrom typing import List, Union\n\n\nclass CLIPEncoder(torch.nn.Module):\n    """\n    Utility class to encode text captions using Microsoft\'s CLIP model.\n    By default, uses \'openai/clip-vit-base-patch32\' which produces 512D embeddings.\n\n    For texts longer than 77 tokens, uses chunk-and-average approach to preserve\n    all text content.\n    """\n\n    def __init__(\n        self,\n        model_name: str = "openai/clip-vit-base-patch32",\n        max_length: int = 77,\n    ):\n        super().__init__()\n\n        self.max_length = max_length\n\n        print(f"Loading CLIP model \'{model_name}\'...")\n        self.tokenizer = CLIPTokenizer.from_pretrained(model_name)\n        self.model = CLIPTextModel.from_pretrained(model_name)\n        self.model.eval()\n\n        # Freeze CLIP parameters\n        for param in self.model.parameters():\n            param.requires_grad = False\n\n    @torch.no_grad()\n    def forward(self, text: Union[str, List[str]]) -> torch.Tensor:\n        """\n        Encode a list of captions or a single caption into embeddings.\n\n        For texts longer than max_length tokens, splits into chunks and averages\n        the embeddings to preserve all text content.\n\n        Args:\n            text: A single string or a list of strings.\n\n        Returns:\n            embeddings: (B, 1, 512) tensor containing pooled CLIP embeddings.\n        """\n        if isinstance(text, str):\n            text = [text]\n\n        # Determine device dynamically\n        device = next(self.model.parameters()).device\n\n        inputs = self.tokenizer(\n            text, padding=True, truncation=True, return_tensors="pt"\n        ).to(device)\n        outputs = self.model(**inputs)\n\n        # Use the pooler_output for a global representation of the sentence\n        # Shape: (Batch_Size, 512)\n        embeddings = outputs.pooler_output.unsqueeze(1)\n\n        return embeddings\n\n    @property\n    def embedding_dim(self) -> int:\n        """Output dimension of the CLIP text model."""\n        return self.model.config.hidden_size\n',
        'utils/wandb_logger.py': "import os\nimport sys\nfrom datetime import datetime\nfrom typing import Optional, Dict, Any\ntry:\n    import wandb\n    WANDB_AVAILABLE = True\nexcept ImportError:\n    WANDB_AVAILABLE = False\n    wandb = None\n\ndef is_kaggle_environment() -> bool:\n    return os.path.exists('/kaggle') or 'kaggle' in sys.executable.lower()\n\ndef get_kaggle_secret(secret_name: str) -> Optional[str]:\n    if not is_kaggle_environment():\n        return None\n    try:\n        from kaggle_secrets import UserSecretsClient\n        user_secrets = UserSecretsClient()\n        return user_secrets.get_secret(secret_name)\n    except Exception:\n        return None\n\nclass WandbLogger:\n\n    def __init__(self, project: str, name: Optional[str]=None, config: Optional[Dict[str, Any]]=None, kaggle_secret_name: str='WANDB_API_KEY', enabled: bool=True):\n        self.project = project\n        self.config = config or {}\n        self.enabled = enabled and WANDB_AVAILABLE\n        self.run = None\n        if name is None:\n            self.name = 'motion-generation-buet'\n        else:\n            self.name = name\n        if not self.enabled:\n            if not WANDB_AVAILABLE:\n                print('[WandbLogger] wandb not installed. Logging disabled.')\n            elif not enabled:\n                print('[WandbLogger] Logging disabled by user.')\n            return\n        self._authenticate(kaggle_secret_name)\n        try:\n            self.run = wandb.init(project=project, entity='motion-generation-buet', config=config, reinit=True)\n            print(f'[WandbLogger] Initialized run: {self.run.name}')\n            print(f'[WandbLogger] View at: {self.run.url}')\n        except Exception as e:\n            print(f'[WandbLogger] Failed to initialize: {e}')\n            self.enabled = False\n\n    def _authenticate(self, secret_name: str) -> None:\n        api_key = os.environ.get('WANDB_API_KEY')\n        if api_key is None:\n            api_key = get_kaggle_secret(secret_name)\n        if api_key:\n            try:\n                wandb.login(key=api_key)\n                print('[WandbLogger] Authenticated successfully.')\n            except Exception as e:\n                print(f'[WandbLogger] Authentication failed: {e}')\n        else:\n            print('[WandbLogger] No API key found. Using existing login or anonymous mode.')\n\n    def log(self, metrics: Dict[str, Any], step: Optional[int]=None) -> None:\n        if not self.enabled or self.run is None:\n            return\n        try:\n            wandb.log(metrics, step=step)\n        except Exception as e:\n            print(f'[WandbLogger] Failed to log metrics: {e}')\n\n    def log_model(self, path: str, name: str, description: Optional[str]=None) -> None:\n        if not self.enabled or self.run is None:\n            return\n        try:\n            artifact = wandb.Artifact(name, type='model', description=description)\n            artifact.add_file(path)\n            self.run.log_artifact(artifact)\n            print(f'[WandbLogger] Logged model artifact: {name}')\n        except Exception as e:\n            print(f'[WandbLogger] Failed to log model: {e}')\n\n    def log_summary(self, metrics: Dict[str, Any]) -> None:\n        if not self.enabled or self.run is None:\n            return\n        try:\n            for key, value in metrics.items():\n                wandb.run.summary[key] = value\n        except Exception as e:\n            print(f'[WandbLogger] Failed to log summary: {e}')\n\n    def finish(self) -> None:\n        if not self.enabled or self.run is None:\n            return\n        try:\n            wandb.finish()\n            print('[WandbLogger] Run finished.')\n        except Exception as e:\n            print(f'[WandbLogger] Failed to finish run: {e}')\n\n    def __enter__(self) -> 'WandbLogger':\n        return self\n\n    def __exit__(self, exc_type, exc_val, exc_tb) -> None:\n        self.finish()",
    }
    
    for filepath, content in FILES.items():
        path = Path(filepath)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, 'w', encoding='utf-8') as f:
            f.write(content)
        print(f'Created {filepath}')
    
    # Install dependencies
    print("Installing dependencies (this may take a minute)...")
    %pip install -r requirements.txt
    
    # Copy dataset
    print("Copying dataset...")
    !apt -qq install rclone && rclone copy /kaggle/input/datasets/mustafamuhaimin/ /kaggle/working/dataset/ --transfers 16 --checkers 16 --progress --ignore-existing -q
    
    print("Setup Complete!")
else:
    print("Running locally. No setup needed.")
